# Studies 1–3: cogency and expected rhetorical effectiveness

Reproduces every table of the thesis analysis from the files in this repository. **No API calls.** Runtime: about 10 minutes on a laptop or Colab.

- **Human annotation:** H1 labeled all 115 main-round posts (1,725 pairs) and is the main human reference. H2 (115 posts) and H3 (58 posts) are used for human–human agreement, ranking reliability and robustness.
- **LLM annotation:** Claude Sonnet 5.5 (primary) and GPT-6.1 Sol (robustness), every pair judged in both A/B orders.
- **Studies:** 1 = human–LLM correspondence (multitrait–multimethod), 2 = behavioral outcomes, 3 = held-out prediction.

Inputs (paths relative to the repository root):
- `data/annotation_input.csv`, `data/outcomes.csv`
- `data/human/main_H{1,2,3}.csv`, `data/human/calibration_H{1,2,3}.csv`
- `llm_annotation/runs/{claude,gpt}/llm_output/llm_annotations_*.csv`

Outputs: every table as `.csv` and `.tex` in `results/tables/`.

## 1. Setup

In [1]:
import glob, hashlib, itertools
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown

B       = 1000   # bootstrap replicates for rankings and outcome models
B_PAIRS = 2000   # bootstrap replicates for pairwise agreement
LAM     = 0.1    # Bradley-Terry regularization

# Repository root: the first folder at or above the working directory that contains data/annotation_input.csv.
# In Colab, run `!git clone <repo-url>` and `%cd <repo-folder>` first.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "annotation_input.csv").exists())
INPUT   = ROOT / "data" / "annotation_input.csv"
V3      = ROOT / "data" / "outcomes.csv"
RUNS    = ROOT / "llm_annotation" / "runs"
H1_FILE = ROOT / "data" / "human" / "main_H1.csv"
H2_FILE = ROOT / "data" / "human" / "main_H2.csv"
H3_FILE = ROOT / "data" / "human" / "main_H3.csv"
CAL     = sorted(glob.glob(str(ROOT / "data" / "human" / "calibration_H*.csv")))
for p in (INPUT, H1_FILE, H2_FILE, H3_FILE, V3):
    assert p.exists(), f"Missing file: {p}"
OUT = ROOT / "results"; T = OUT / "tables"
T.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Calibration files:", len(CAL))

Repository: cmv-cogency-effectiveness
Calibration files: 3


## 2. Analysis functions (run this cell; it prints nothing)

In [2]:
"""
Studies 1-4 of the thesis proposal, computed from existing human and LLM pairwise annotations
plus the behavioral outcomes saved with the human-validation sample. No API calls.

Only numpy, pandas and scipy are required (no statsmodels): the conditional logit and the
fixed-effect regressions are implemented here so the notebook runs anywhere.

Constructs: cogency (C) and expected rhetorical effectiveness (E). Evidential support is not
part of Studies 1-4 in the proposal and is left out.
"""
from __future__ import annotations
import itertools
import numpy as np
import pandas as pd
from scipy import optimize, stats

CONSTRUCTS = {"C": ("cogency", "cogency"), "E": ("effectiveness", "effect")}
METHODS = ("H", "gpt", "claude")
METHOD_NAMES = {"H": "Human", "gpt": "GPT-6.1 Sol", "claude": "Claude Sonnet 5.5"}


# =============================================================================== measurement
def judgments(human: pd.DataFrame, llm: dict) -> pd.DataFrame:
    """Long table of pairwise judgments: post_id, lo, hi, construct, method, orientation,
    s (canonical signed label: negative = lo preferred), winner, loser."""
    rows = []
    for t, (_, pre) in CONSTRUCTS.items():
        h = human[["post_id", "lo", "hi", f"{pre}_signed_canon"]].rename(columns={f"{pre}_signed_canon": "s"})
        h = h.assign(construct=t, method="H", orientation="human")
        rows.append(h)
        for m in ("gpt", "claude"):
            d = llm[(m, t)]
            parts = d.pair_key.str.split("|", expand=True)
            rows.append(pd.DataFrame({"post_id": parts[0], "lo": parts[1], "hi": parts[2],
                                      "s": pd.to_numeric(d[f"{pre}_signed_canon"]), "construct": t,
                                      "method": m, "orientation": d.orientation.values}))
    J = pd.concat(rows, ignore_index=True).dropna(subset=["s"])
    J["s"] = J.s.astype(float)
    J["winner"] = np.where(J.s < 0, J.lo, J.hi)
    J["loser"] = np.where(J.s < 0, J.hi, J.lo)
    return J


def bradley_terry(items, wins, lam=0.1, weights=None):
    """Regularized Bradley-Terry: maximize sum w*log sigmoid(theta_w - theta_l) - lam*sum theta^2.
    items: list of ids; wins: list of (winner, loser)."""
    idx = {r: i for i, r in enumerate(items)}
    W = np.array([idx[a] for a, _ in wins]); Lo = np.array([idx[b] for _, b in wins])
    w = np.ones(len(wins)) if weights is None else np.asarray(weights, float)
    n = len(items)

    def f(th):
        d = th[W] - th[Lo]
        ll = np.sum(w * -np.logaddexp(0, -d)) - lam * np.sum(th ** 2)
        p = 1 / (1 + np.exp(d))            # = 1 - sigmoid(d)
        g = -2 * lam * th
        np.add.at(g, W, w * p); np.add.at(g, Lo, -w * p)
        return -ll, -g
    res = optimize.minimize(f, np.zeros(n), jac=True, method="L-BFGS-B")
    return dict(zip(items, res.x))


def within_op_percentile(x: pd.Series) -> pd.Series:
    r = x.rank(method="average")
    return (r - 0.5) / len(x)


def scores(J: pd.DataFrame, responses: pd.DataFrame, lam=0.1, variant="bt") -> pd.DataFrame:
    """One row per response x method x construct: BT score (theta) and within-OP percentile (q).
    variant 'bt' uses winners only; 'bt_strength' counts Strong judgments twice; 'signed' sums
    signed labels (Borda-like)."""
    out = []
    items_by_post = responses.groupby("post_id").response_id.apply(list).to_dict()
    for (pid, m, t), g in J.groupby(["post_id", "method", "construct"]):
        items = items_by_post[pid]
        if variant == "signed":
            sc = {r: 0.0 for r in items}
            for lo, hi, s in zip(g.lo, g.hi, g.s):
                sc[lo] -= s; sc[hi] += s
        else:
            w = np.where(np.abs(g.s) == 2, 2.0, 1.0) if variant == "bt_strength" else None
            sc = bradley_terry(items, list(zip(g.winner, g.loser)), lam, w)
        s = pd.Series(sc)
        out.append(pd.DataFrame({"post_id": pid, "response_id": s.index, "method": m, "construct": t,
                                 "theta": s.values, "q": within_op_percentile(s).values}))
    return pd.concat(out, ignore_index=True)


def wide(S: pd.DataFrame, value="q") -> pd.DataFrame:
    """response_id x (method_construct) table, e.g. columns H_C, H_E, gpt_C, ..."""
    w = S.pivot_table(index=["post_id", "response_id"], columns=["method", "construct"], values=value)
    w.columns = [f"{m}_{t}" for m, t in w.columns]
    return w.reset_index()


# =============================================================================== associations
def kendall_tau_b_by_op(W: pd.DataFrame, a: str, b: str) -> pd.Series:
    """Per-OP Kendall tau-b between two measurement columns."""
    def tau(g):
        if g[a].nunique() < 2 or g[b].nunique() < 2:
            return np.nan
        return stats.kendalltau(g[a], g[b]).statistic
    return W.groupby("post_id")[[a, b]].apply(tau)


def assoc(W, a, b, kind="tau"):
    """Association between two measurements: mean within-OP Kendall tau-b ('tau', the proposal's
    two-stage estimator) or pooled Spearman on within-OP percentiles ('rho')."""
    if kind == "tau":
        return kendall_tau_b_by_op(W, a, b).mean()
    d = W[[a, b]].dropna()
    return d[a].rank().corr(d[b].rank())


def op_bootstrap(W, stat, B=1000, seed=0):
    """Percentile bootstrap over OPs. stat(W) -> float or dict of floats."""
    rng = np.random.default_rng(seed)
    groups = {p: g for p, g in W.groupby("post_id")}
    ids = np.array(list(groups))
    est = stat(W)
    keys = list(est) if isinstance(est, dict) else None
    reps = []
    for b in range(B):
        pick = rng.choice(ids, len(ids), replace=True)
        Wb = pd.concat([groups[p].assign(post_id=f"{p}#{i}") for i, p in enumerate(pick)], ignore_index=True)
        reps.append(stat(Wb))
    if keys is None:
        v = np.array(reps, float); v = v[~np.isnan(v)]
        return est, np.percentile(v, 2.5), np.percentile(v, 97.5), boot_p(v)
    out = {}
    for k in keys:
        v = np.array([r[k] for r in reps], float); v = v[~np.isnan(v)]
        out[k] = (est[k], np.percentile(v, 2.5), np.percentile(v, 97.5), boot_p(v))
    return out


def boot_p(v):
    """Two-sided bootstrap p-value for H0: value = 0."""
    if len(v) == 0:
        return np.nan
    return min(1.0, 2 * min(np.mean(v <= 0), np.mean(v >= 0)) + 1 / len(v))


def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); running = 0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1, running)
    return adj


def split_half_reliability(J, responses, lam=0.1):
    """LLM reliability from orientation: BT ranking from A/B-order judgments only vs. from
    B/A-order judgments only, per OP (mean Kendall tau-b), then Spearman-Brown for the full set."""
    out = {}
    for m in ("gpt", "claude"):
        for t in CONSTRUCTS:
            sub = J[(J.method == m) & (J.construct == t)]
            Sa = scores(sub[sub.orientation == "AB"], responses, lam)
            Sb = scores(sub[sub.orientation == "BA"], responses, lam)
            Wd = Sa[["post_id", "response_id", "q"]].merge(Sb[["post_id", "response_id", "q"]],
                                                         on=["post_id", "response_id"], suffixes=("_a", "_b"))
            r_half = kendall_tau_b_by_op(Wd, "q_a", "q_b").mean()
            rho_half = Wd.q_a.rank().corr(Wd.q_b.rank())
            out[(m, t)] = {"tau_half": r_half, "rho_half": rho_half,
                           "rho_full_sb": 2 * rho_half / (1 + rho_half)}
    return out


def human_reliability(cal_files, lam=0.1):
    """Human-human agreement of per-OP rankings on calibration OPs (several annotators, same pairs).
    Returns mean pairwise Kendall tau-b and pooled Spearman per construct, or {} if unavailable."""
    frames = []
    for f in cal_files:
        d = pd.read_csv(f, dtype=str)
        need = {"pair_key", "annotator_id", "response_id_a", "cogency_signed", "effect_signed"}
        if not need <= set(d.columns):
            continue
        p = d.pair_key.str.split("|", expand=True)
        d["post_id"], d["lo"], d["hi"] = p[0], p[1], p[2]
        flip = d.response_id_a != d.lo
        for t, (_, pre) in CONSTRUCTS.items():
            s = pd.to_numeric(d[f"{pre}_signed"], errors="coerce")
            frames.append(pd.DataFrame({"post_id": d.post_id, "lo": d.lo, "hi": d.hi, "construct": t,
                                        "method": d.annotator_id, "orientation": "human",
                                        "s": np.where(flip, -s, s)}))
    if not frames:
        return {}
    J = pd.concat(frames, ignore_index=True).dropna(subset=["s"])
    J["winner"] = np.where(J.s < 0, J.lo, J.hi); J["loser"] = np.where(J.s < 0, J.hi, J.lo)
    resp = pd.DataFrame({"post_id": np.r_[J.post_id, J.post_id], "response_id": np.r_[J.lo, J.hi]}).drop_duplicates()
    S = scores(J, resp, lam)
    Wd = wide(S)
    anns = sorted(J.method.unique())
    out = {"annotators": anns, "ops": J.post_id.nunique()}
    for t in CONSTRUCTS:
        taus, rhos = [], []
        for a, b in itertools.combinations(anns, 2):
            ca, cb = f"{a}_{t}", f"{b}_{t}"
            if ca in Wd and cb in Wd:
                taus.append(kendall_tau_b_by_op(Wd, ca, cb).mean())
                rhos.append(Wd[[ca, cb]].dropna().rank().corr().iloc[0, 1])
        out[t] = {"tau_pairwise_mean": np.nanmean(taus) if taus else np.nan,
                  "rho_pairwise_mean": np.nanmean(rhos) if rhos else np.nan, "pairs_of_annotators": len(taus)}
    return out


# =============================================================================== outcome models
def _within(X, groups, w=None):
    """Subtract (weighted) group means from each column of X."""
    X = np.asarray(X, float)
    w = np.ones(len(X)) if w is None else np.asarray(w, float)
    codes = pd.factorize(pd.Series(groups).values)[0]
    sw = np.bincount(codes, weights=w)
    means = np.column_stack([np.bincount(codes, weights=w * X[:, j]) / sw for j in range(X.shape[1])])
    return X - means[codes]


def fe_ols(y, X, groups, w=None):
    """Linear model with OP fixed effects (within transformation), optional weights,
    cluster-robust (by OP) covariance. Returns beta, cov."""
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    Xd = _within(X, groups, w); yd = _within(np.asarray(y, float)[:, None], groups, w)[:, 0]
    XtX = (Xd * w[:, None]).T @ Xd
    beta = np.linalg.solve(XtX, (Xd * w[:, None]).T @ yd)
    e = yd - Xd @ beta
    G = pd.Series(groups).values; ug = np.unique(G)
    meat = np.zeros((X.shape[1], X.shape[1]))
    for gg in ug:
        i = G == gg
        sg = (Xd[i] * (w[i] * e[i])[:, None]).sum(0)
        meat += np.outer(sg, sg)
    n, k, c = len(y), X.shape[1], len(ug)
    corr = c / (c - 1) * (n - 1) / (n - k - c) if n - k - c > 0 else c / (c - 1)
    Ainv = np.linalg.inv(XtX)
    return beta, corr * Ainv @ meat @ Ainv


def clogit(y, X, groups):
    """Exact conditional logit (one stratum per OP; strata without variation in y are dropped).
    Each stratum's likelihood enumerates all subsets with the observed number of events, which is
    cheap for six responses. Returns beta, cov (inverse observed information).
    Coefficients with |beta| > 15 indicate (quasi-)separation and should not be interpreted."""
    y = np.asarray(y, int); X = np.asarray(X, float)
    codes = pd.factorize(pd.Series(groups).values)[0]
    p = X.shape[1]
    obs, sets = [], []
    order = np.argsort(codes, kind="stable")
    bounds = np.flatnonzero(np.diff(np.r_[-1, codes[order], -2]))
    for s0, s1 in zip(bounds[:-1], bounds[1:]):
        i = order[s0:s1]
        k = y[i].sum()
        if k == 0 or k == len(i):
            continue
        obs.append(X[i[y[i] == 1]].sum(0))
        sets.append(np.array([X[list(c)].sum(0) for c in itertools.combinations(i, k)]))
    if not obs:
        return np.full(p, np.nan), np.full((p, p), np.nan)
    m = max(len(S) for S in sets)
    T = np.zeros((len(sets), m, p)); mask = np.zeros((len(sets), m), bool)
    for j, S in enumerate(sets):
        T[j, :len(S)] = S; mask[j, :len(S)] = True
    xo = np.array(obs).sum(0)

    def nll(b, need=True):
        u = np.where(mask, T @ b, -np.inf)
        mx = u.max(1, keepdims=True); e = np.exp(u - mx); Z = e.sum(1, keepdims=True)
        val = -(xo @ b - np.sum(mx[:, 0] + np.log(Z[:, 0])))
        if not need:
            return val
        pr = e / Z
        mu = np.einsum("sm,smp->sp", pr, T)
        grad = -(xo - mu.sum(0))
        hess = np.einsum("sm,smp,smq->pq", pr, T, T) - mu.T @ mu
        return val, grad, hess

    b = np.zeros(p)
    for _ in range(100):
        v, g, H = nll(b)
        step = np.linalg.lstsq(H + 1e-9 * np.eye(p), g, rcond=None)[0]
        t = 1.0
        while nll(b - t * step, False) > v + 1e-12 and t > 1e-6:
            t /= 2
        b = b - t * step
        if np.max(np.abs(t * step)) < 1e-8 or np.max(np.abs(b)) > 50:
            break
    _, _, H = nll(b)
    return b, np.linalg.pinv(H)


def clogit_scores(beta, X):
    return np.asarray(X, float) @ beta


def wald(beta, cov, idx):
    b = beta[idx]; V = cov[np.ix_(idx, idx)]
    stat = float(b @ np.linalg.pinv(V) @ b)
    return stat, stats.chi2.sf(stat, len(idx))


def covariates(D: pd.DataFrame) -> pd.DataFrame:
    """Prespecified covariates: log word count, elapsed time (log hours and its square, a simple
    non-linear form), log response order."""
    lh = np.log1p(D.hours_since_op.clip(lower=0))
    return pd.DataFrame({"log_words": np.log1p(D.n_words), "log_hours": lh, "log_hours_sq": lh ** 2,
                         "log_order": np.log(D.response_order.clip(lower=1))}, index=D.index)


def design(D, c, e, param="separate"):
    X = covariates(D)
    X = X.loc[:, X.std() > 1e-9]          # drop a covariate with no variation (would make the model singular)
    if param == "separate":
        X.insert(0, "e", D[e].values); X.insert(0, "c", D[c].values)
    else:  # level/difference reparameterization
        X.insert(0, "diff", ((D[e] - D[c]) / 2).values); X.insert(0, "level", ((D[c] + D[e]) / 2).values)
    return X


def prepare(D, c, e, param="separate", weights=False):
    """Numeric arrays for the Study 2 models (built once, reused by every bootstrap replicate)."""
    X = design(D, c, e, param)
    return {"X": X.values.astype(float), "cols": list(X.columns), "state": D.state.values.astype(str),
            "U": D.U.values.astype(float), "post": pd.factorize(D.post_id.values)[0],
            "w": (1 / D.inclusion_prob_within_op.values.astype(float)) if weights else None}


def fit_prepared(P, idx=None, gid=None):
    """Fit the four outcome models on rows idx (default all) with group ids gid."""
    idx = np.arange(len(P["U"])) if idx is None else idx
    g = P["post"][idx] if gid is None else gid
    X, st, cols = P["X"][idx], P["state"][idx], P["cols"]
    res = {}
    for name, keep, pos in (("D", np.ones(len(idx), bool), "D"), ("Rstar_vs_N", st != "D", "R"),
                            ("D_vs_N", st != "R", "D")):
        b, V = clogit((st[keep] == pos).astype(int), X[keep], g[keep])
        res[name] = (pd.Series(b, cols), pd.DataFrame(V, cols, cols))
    w = None if P["w"] is None else P["w"][idx]
    b, V = fe_ols(P["U"][idx], X, g, w)
    res["U"] = (pd.Series(b, cols), pd.DataFrame(V, cols, cols))
    return res


def fit_outcomes(D, c, e, param="separate", weights=False):
    """Study 2 models on dataset D with construct columns c, e.
    D: conditional logit for delta (all responses).  R*: conditional logit R* vs N (D excluded).
    D vs N: conditional logit (R* excluded).  U: within-OP score percentile, OP fixed effects.
    Returns {outcome: (beta Series, cov DataFrame)}."""
    return fit_prepared(prepare(D, c, e, param, weights))


def boot_indices(D, B, seed):
    """Yield (row index, replicate group id) arrays for an OP bootstrap."""
    rng = np.random.default_rng(seed)
    codes = pd.factorize(D.post_id.values)[0]
    rows_of = [np.flatnonzero(codes == k) for k in range(codes.max() + 1)]
    n = len(rows_of)
    for _ in range(B):
        pick = rng.integers(0, n, n)
        idx = np.concatenate([rows_of[k] for k in pick])
        gid = np.repeat(np.arange(n), [len(rows_of[k]) for k in pick])
        yield idx, gid


def boot_outcomes(D, c, e, B=1000, seed=0, param="separate", weights=False, extra=None):
    """OP bootstrap of fit_outcomes coefficients (construct terms). extra(res)->dict adds
    derived statistics (e.g. differential-prediction angle)."""
    P = prepare(D, c, e, param, weights)
    keys = ["c", "e"] if param == "separate" else ["level", "diff"]
    out = []
    for idx, gid in boot_indices(D, B, seed):
        try:
            r = fit_prepared(P, idx, gid)
        except np.linalg.LinAlgError:
            continue
        row = {f"{o}:{k}": r[o][0][k] for o in r for k in keys}
        if extra:
            row.update(extra(r))
        out.append(row)
    return pd.DataFrame(out)


def angle(bD, bU):
    """Signed angle (degrees) between construct-coefficient vectors (beta_C, beta_E) of two
    outcomes. 0 = proportional (same relative weighting)."""
    a = np.arctan2(bD[1], bD[0]); b = np.arctan2(bU[1], bU[0])
    d = (a - b + np.pi) % (2 * np.pi) - np.pi
    d = (d + np.pi / 2) % np.pi - np.pi / 2       # a vector and its negative have the same weighting
    return np.degrees(d)


def heldout_discrimination(D, c, e, folds=5, seed=0):
    """Held-out-OP evaluation of nested models. Delta: within-OP AUC (share of D vs non-D response
    pairs in the same OP where the D response scores higher). U: within-OP Spearman between
    predicted and observed U. Returns per-OP metrics for each model."""
    rng = np.random.default_rng(seed)
    ops = np.array(sorted(D.post_id.unique())); rng.shuffle(ops)
    fold_of = {p: i % folds for i, p in enumerate(ops)}
    models = {"covariates only": [], "cogency + covariates": ["c"], "effectiveness + covariates": ["e"],
              "both + covariates": ["c", "e"]}
    rows = []
    for k in range(folds):
        tr = D[D.post_id.map(fold_of) != k]; te = D[D.post_id.map(fold_of) == k]
        for name, cons in models.items():
            def X_of(d):
                X = covariates(d)
                X = X.loc[:, covariates(D).std() > 1e-9]
                for col in cons:
                    X.insert(0, col, d[{"c": c, "e": e}[col]].values)
                return X
            b, _ = clogit(tr.y_D.values, X_of(tr).values, tr.post_id.values)
            bu, _ = fe_ols(tr.U.values, X_of(tr).values, tr.post_id.values)
            te = te.assign(pD=X_of(te).values @ b, pU=X_of(te).values @ bu)
            for pid, g in te.groupby("post_id"):
                pos, neg = g[g.y_D == 1].pD.values, g[g.y_D == 0].pD.values
                auc = np.mean([(p > n) + 0.5 * (p == n) for p in pos for n in neg]) if len(pos) and len(neg) else np.nan
                rhoU = g.pU.rank().corr(g.U.rank()) if g.U.nunique() > 1 else np.nan
                rows.append({"post_id": pid, "model": name, "auc_D": auc, "rho_U": rhoU})
    return pd.DataFrame(rows)


# =============================================================================== discordance
def pair_winners(J):
    """Per pair x method x construct: majority winner sign (LLM: over both orders; 0 = split)."""
    g = J.groupby(["post_id", "lo", "hi", "method", "construct"]).s.apply(lambda s: np.sign(np.sign(s).sum()))
    return g.unstack("construct").reset_index()


def discordance(J):
    """Z = 1 if the method picks different winners for cogency and effectiveness on the pair
    (pairs where an LLM's two orders disagree on either construct are reported as split)."""
    P = pair_winners(J)
    P["split"] = (P.C == 0) | (P.E == 0)
    P["Z"] = np.where(P.split, np.nan, (P.C != P.E).astype(float))
    return P


# =============================================================================== data assembly
def build_dataset(outcomes: pd.DataFrame, W: pd.DataFrame, text: pd.DataFrame) -> pd.DataFrame:
    """Response-level analysis table: measurements (W, wide within-OP percentiles), behavioral
    outcomes from the sampling file, and word counts of the text shown to annotators/models."""
    o = outcomes.copy()
    o["state"] = o.behavior_state_star.astype(str)
    o["y_D"] = (o.state == "D").astype(int)
    o["U"] = pd.to_numeric(o.community_score_percentile)
    o["hours_since_op"] = pd.to_numeric(o.hours_since_op)
    o["response_order"] = pd.to_numeric(o.response_order)
    o["inclusion_prob_within_op"] = pd.to_numeric(o.inclusion_prob_within_op)
    keep = ["post_id", "response_id", "state", "y_D", "U", "hours_since_op", "response_order",
            "inclusion_prob_within_op", "behavior_state", "op_stratum"]
    t = text[["response_id", "response_text_clean"]].copy()
    t["n_words"] = t.response_text_clean.fillna("").str.split().str.len()
    D = W.merge(o[keep], on=["post_id", "response_id"], how="inner").merge(t[["response_id", "n_words"]], on="response_id")
    return D


# =============================================================================== fast Study 1 statistics
def op_matrices(W: pd.DataFrame, cols) -> tuple[np.ndarray, dict]:
    """(post_ids, {col: array (n_OP x 6)}) with responses sorted by id within each OP.
    OPs missing any of cols are dropped."""
    d = W.dropna(subset=list(cols)).sort_values(["post_id", "response_id"])
    sizes = d.groupby("post_id").size()
    keep = sizes[sizes == 6].index
    d = d[d.post_id.isin(keep)]
    return np.array(sorted(keep)), {c: d[c].to_numpy().reshape(-1, 6) for c in cols}


_IU = np.triu_indices(6, 1)

def tau_b_rows(a, b):
    """Kendall tau-b for each row of a and b (n x 6)."""
    da = np.sign(a[:, :, None] - a[:, None, :])[:, _IU[0], _IU[1]]
    db = np.sign(b[:, :, None] - b[:, None, :])[:, _IU[0], _IU[1]]
    num = (da * db).sum(1)
    den = np.sqrt((da != 0).sum(1) * (db != 0).sum(1))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def pooled_r(a, b):
    """Pearson correlation of within-OP percentiles pooled over OPs (a rank correlation, since
    the percentiles are within-OP ranks)."""
    x, y = a.ravel(), b.ravel()
    return np.corrcoef(x, y)[0, 1]


def A(M, a, b, kind):
    return np.nanmean(tau_b_rows(M[a], M[b])) if kind == "tau" else pooled_r(M[a], M[b])


def study1_stats(M, kind="tau", models=("gpt", "claude")):
    out = {}
    for m in models:
        out[f"A|H_C|{m}_C"] = A(M, "H_C", f"{m}_C", kind)
        out[f"A|H_C|{m}_E"] = A(M, "H_C", f"{m}_E", kind)
        out[f"A|H_E|{m}_E"] = A(M, "H_E", f"{m}_E", kind)
        out[f"A|H_E|{m}_C"] = A(M, "H_E", f"{m}_C", kind)
        out[f"A|{m}_C|{m}_E"] = A(M, f"{m}_C", f"{m}_E", kind)
        out[f"Delta_C|{m}"] = out[f"A|H_C|{m}_C"] - out[f"A|H_C|{m}_E"]
        out[f"Delta_E|{m}"] = out[f"A|H_E|{m}_E"] - out[f"A|H_E|{m}_C"]
    out["A|H_C|H_E"] = A(M, "H_C", "H_E", kind)
    for m in models:
        out[f"Gamma|{m}"] = out[f"A|{m}_C|{m}_E"] - out["A|H_C|H_E"]
    if len(models) == 2:
        a, b = models
        out[f"A|{a}_C|{b}_C"] = A(M, f"{a}_C", f"{b}_C", kind)
        out[f"A|{a}_E|{b}_E"] = A(M, f"{a}_E", f"{b}_E", kind)
    return out


def lambda_stats(M, methods=("H", "gpt", "claude")):
    """Audience-specificity increment 1 - R^2(E ~ C) on within-OP percentiles (observed)."""
    return {f"Lambda|{m}": 1 - pooled_r(M[f"{m}_C"], M[f"{m}_E"]) ** 2 for m in methods}


def boot_matrix_stats(M, stat, B=1000, seed=0):
    """OP bootstrap for statistics computed from op_matrices output."""
    rng = np.random.default_rng(seed)
    n = next(iter(M.values())).shape[0]
    est = stat(M)
    reps = {k: [] for k in est}
    for _ in range(B):
        i = rng.integers(0, n, n)
        r = stat({k: v[i] for k, v in M.items()})
        for k in est:
            reps[k].append(r[k])
    out = {}
    for k in est:
        v = np.array(reps[k], float); v = v[~np.isnan(v)]
        out[k] = (est[k], np.percentile(v, 2.5), np.percentile(v, 97.5), boot_p(v))
    return out


# ---- table output (CSV + LaTeX booktabs), same style as the discussion tables
def tex_escape(x):
    s = str(x)
    s = s.replace("\\", "\x00")
    for a, b in (("{", r"\{"), ("}", r"\}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"), ("$", r"\$"),
                 ("^", r"\textasciicircum{}"), ("~", r"\textasciitilde{}"), (">=", r"$\geq$"), ("<", r"\textless{}"), (">", r"\textgreater{}")):
        s = s.replace(a, b)
    return s.replace("\x00", r"\textbackslash{}")

ALL = []
def save(df, name, caption, note=""):
    df.to_csv(T / f"{name}.csv", index=False)
    cols = list(df.columns)
    spec = "l" * min(2, len(cols)) + "r" * (len(cols) - min(2, len(cols)))
    body = [" & ".join(tex_escape(c) for c in cols) + r" \\", r"\midrule"]
    body += [" & ".join(tex_escape(v) for v in row) + r" \\" for row in df.astype(str).values]
    tab = "\n".join([r"\begin{tabular}{" + spec + "}", r"\toprule", *body, r"\bottomrule", r"\end{tabular}"])
    if len(cols) >= 5:
        tab = r"\resizebox{\ifdim\width>\textwidth\textwidth\else\width\fi}{!}{%" + "\n" + tab + "}"
    tex = "\n".join([r"\begin{table}[htbp]", r"\centering", r"\small", r"\caption{" + tex_escape(caption) + "}",
                     r"\label{tab:" + name + "}", tab] +
                    ([r"\par\vspace{2pt}\footnotesize\raggedright " + tex_escape(note)] if note else []) +
                    [r"\end{table}", ""])
    (T / f"{name}.tex").write_text(tex, encoding="utf-8")
    ALL.append(name)
    (T / "all_tables.tex").write_text("\n".join(r"\input{tables/" + s + "}" for s in sorted(set(ALL))) + "\n")
    display(Markdown(f"**{name}. {caption}**" + (f"  \n_{note}_" if note else "")))
    display(df)

def f2(x):  return "--" if x is None or pd.isna(x) else f"{x:.2f}"
def f3(x):  return "--" if x is None or pd.isna(x) else f"{x:.3f}"
def ci(t, d=2):
    e, lo, hi = t[:3]
    return "--" if pd.isna(e) else f"{e:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"
def pval(p): return "--" if pd.isna(p) else ("<.001" if p < 0.001 else f"{p:.3f}")


# =============================================================================== changes for this notebook
def within_op_percentile(x: pd.Series) -> pd.Series:
    """Within-post percentile. Scores are rounded first so that responses with identical
    Bradley-Terry scores (identical win records) are treated as tied instead of being
    ordered by optimizer rounding noise."""
    r = x.round(6).rank(method="average")
    return (r - 0.5) / len(x)


SIGNED = {"Strong A": -2, "Weak A": -1, "Weak B": 1, "Strong B": 2}
ANN_NAMES = {"H1": "H1", "H2": "H2", "H3": "H3"}

def canon_human(df):
    """Raw annotation CSV -> canonical labels (negative = lower response id preferred)."""
    df = df.copy()
    p = df.pair_key.str.split("|", expand=True)
    df["post_id"], df["lo"], df["hi"] = p[0], p[1], p[2]
    flip = df.response_id_a != df.lo
    for pre in ("cogency", "effect"):
        s = df[f"{pre}_choice"].map(SIGNED)
        assert s.notna().all(), f"Missing or unknown {pre} labels"
        win = np.where(df[f"{pre}_choice"].str.endswith("A"), df.response_id_a, df.response_id_b)
        assert (win == df[f"{pre}_winner_response_id"]).all(), f"{pre}: winner id does not match the label"
        df[f"{pre}_signed_canon"] = np.where(flip, -s, s).astype(float)
    return df

def kripp_alpha_ordinal(x, y):
    vals = np.array(sorted(set(np.r_[x, y]))); k = {v: i for i, v in enumerate(vals)}
    o = np.zeros((len(vals), len(vals)))
    for a, b in zip(x, y):
        o[k[a], k[b]] += 1; o[k[b], k[a]] += 1
    nc = o.sum(1); n = nc.sum(); d2 = np.zeros_like(o)
    for c in range(len(vals)):
        for e in range(len(vals)):
            lo_, hi_ = min(c, e), max(c, e)
            d2[c, e] = (nc[lo_:hi_ + 1].sum() - (nc[c] + nc[e]) / 2) ** 2
    return 1 - ((o * d2).sum() / n) / ((np.outer(nc, nc) * d2).sum() / (n * (n - 1)))

def cohen_kappa(x, y):
    x, y = np.asarray(x), np.asarray(y); cats = np.union1d(x, y)
    po = np.mean(x == y); pe = sum(np.mean(x == c) * np.mean(y == c) for c in cats)
    return (po - pe) / (1 - pe)

def weighted_kappa(x, y, cats=(-2, -1, 1, 2)):
    idx = {c: i for i, c in enumerate(cats)}; k = len(cats); O = np.zeros((k, k))
    for a, b in zip(x, y): O[idx[a], idx[b]] += 1
    O /= O.sum(); E = np.outer(O.sum(1), O.sum(0))
    Wt = np.abs(np.subtract.outer(np.arange(k), np.arange(k))) / (k - 1)
    return 1 - (Wt * O).sum() / (Wt * E).sum()

def boot_df(df, stat, B=1000, seed=0):
    """Point estimate and 95% percentile CI, resampling posts (all rows of a post together)."""
    rng = np.random.default_rng(seed)
    df = df.reset_index(drop=True)
    idx = list(df.groupby("post_id").indices.values())
    est = stat(df); reps = {k: [] for k in est}
    for _ in range(B):
        rows = np.concatenate([idx[i] for i in rng.integers(0, len(idx), len(idx))])
        r = stat(df.iloc[rows])
        for k in est: reps[k].append(r[k])
    return {k: (est[k], np.nanpercentile(reps[k], 2.5), np.nanpercentile(reps[k], 97.5)) for k in est}

def pct(t, d=1):
    return f"{100*t[0]:.{d}f}% [{100*t[1]:.{d}f}, {100*t[2]:.{d}f}]"

def llm_long(llm):
    """One row per LLM judgment: model, construct, pair_key, post_id, lo, hi, orientation, first-shown id, s."""
    rows = []
    for (m, t), d in llm.items():
        p = d.pair_key.str.split("|", expand=True)
        rows.append(pd.DataFrame({"model": m, "construct": t, "pair_key": d.pair_key, "post_id": p[0], "lo": p[1], "hi": p[2],
                                  "orientation": d.orientation, "shown_first": d.response_id_a,
                                  "s": pd.to_numeric(d[f"{CONSTRUCTS[t][1]}_signed_canon"])}))
    return pd.concat(rows, ignore_index=True).dropna(subset=["s"])

def human_long(h):
    rows = []
    for t, (_, pre) in CONSTRUCTS.items():
        rows.append(pd.DataFrame({"construct": t, "pair_key": h.pair_key, "post_id": h.post_id, "h": h[f"{pre}_signed_canon"]}))
    return pd.concat(rows, ignore_index=True)

def human_rankings(humans, responses):
    """Wide table of within-post percentiles for each human: columns <ann>_C, <ann>_E."""
    frames = []
    for a, h in humans.items():
        for t, (_, pre) in CONSTRUCTS.items():
            frames.append(pd.DataFrame({"post_id": h.post_id, "lo": h.lo, "hi": h.hi, "construct": t,
                                        "method": a, "orientation": "human", "s": h[f"{pre}_signed_canon"]}))
    J = pd.concat(frames, ignore_index=True)
    J["winner"] = np.where(J.s < 0, J.lo, J.hi); J["loser"] = np.where(J.s < 0, J.hi, J.lo)
    return wide(scores(J, responses, LAM))

## 3. Load and check the data

Every human file is checked: one label per pair, labels consistent with the winner IDs, and the same pairs the models judged.

In [3]:
EXPECTED_SHA = "292cb43b9d93598aa92d719d3f82c0d0f3ce58e4ba4cf0cfd99d3efdebf8a00d"   # data/outcomes.csv as written by sampling/scrape_and_sample.ipynb
print("Outcomes file matches the released data/outcomes.csv:", hashlib.sha256(V3.read_bytes()).hexdigest() == EXPECTED_SHA)

inp = pd.read_csv(INPUT, dtype=str, encoding="utf-8-sig")
v3  = pd.read_csv(V3, dtype=str, encoding="utf-8-sig")
assert set(inp.response_id) == set(v3.response_id), "Outcome file and annotation input have different responses"
responses = inp[["post_id", "response_id"]]

llm = {(mod, t): pd.read_csv(RUNS / mod / "llm_output" / f"llm_annotations_{name}.csv", dtype=str, encoding="utf-8-sig")
       for mod in ("gpt", "claude") for t, (name, _) in CONSTRUCTS.items()}
MAIN_OPS = set(llm[("gpt", "C")].pair_key.str.split("|").str[0])
LL = llm_long(llm)

H = {k: canon_human(pd.read_csv(p, dtype={"pair_key": str}, keep_default_na=False))
     for k, p in (("H1", H1_FILE), ("H2", H2_FILE), ("H3", H3_FILE))}

llm_pairs = set(LL.pair_key)
for a, h in H.items():
    assert h.pair_key.is_unique, f"{a}: duplicate pairs"
    assert set(h.pair_key) <= llm_pairs, f"{a}: pairs not judged by the models"
    print(f"{ANN_NAMES[a]:10s}: {len(h):5d} pairs, {h.post_id.nunique():3d} posts, all checks passed")
H3_OPS = set(H["H3"].post_id)
assert set(H["H1"].post_id) == MAIN_OPS, "H1 should cover all 115 main-round posts"

Outcomes file matches the released data/outcomes.csv: True
H1        :  1725 pairs, 115 posts, all checks passed
H2        :  1725 pairs, 115 posts, all checks passed
H3        :   870 pairs,  58 posts, all checks passed


## Part A. The three human annotators

**A1.** How each annotator used the scale. **A2.** How often two annotators agree on the same pairs. **A3.** How reliable one annotator's within-post ranking is (agreement between two annotators' rankings).

In [4]:
rows = []
for a, h in H.items():
    c, e = h.cogency_signed_canon, h.effect_signed_canon
    secs = pd.to_numeric(h.get("seconds_on_pair_total"), errors="coerce")
    rows.append({"Annotator": ANN_NAMES[a], "Posts": h.post_id.nunique(), "Pairs": len(h),
                 "Different C and E winners": f"{100*np.mean(np.sign(c) != np.sign(e)):.1f}%",
                 "Identical C and E label": f"{100*np.mean(c == e):.1f}%",
                 "Strong (C / E)": f"{100*np.mean(c.abs() == 2):.0f}% / {100*np.mean(e.abs() == 2):.0f}%",
                 "Median seconds per pair": f"{secs.median():.0f}" if secs.notna().any() and secs.median() > 0 else "--"})
A1 = pd.DataFrame(rows)
save(A1, "A1_annotators", "How each human annotator used the scale",
     "Different C and E winners: share of pairs where the cogency and effectiveness choices pick different responses.")

HH = {}
rows = []
for a, b in (("H1", "H3"), ("H1", "H2"), ("H3", "H2")):
    j = H[a][["pair_key", "post_id", "cogency_signed_canon", "effect_signed_canon"]].merge(
        H[b][["pair_key", "cogency_signed_canon", "effect_signed_canon"]], on="pair_key", suffixes=("_x", "_y"))
    for t, (lab, pre) in CONSTRUCTS.items():
        x, y = f"{pre}_signed_canon_x", f"{pre}_signed_canon_y"
        st = lambda d, x=x, y=y: {"same": np.mean(np.sign(d[x]) == np.sign(d[y])), "alpha": kripp_alpha_ordinal(d[x].values, d[y].values),
                                  "wk": weighted_kappa(d[x].values, d[y].values)}
        r = boot_df(j, st, B_PAIRS, seed=1)
        HH[(a, b, t)] = r
        rows.append({"Annotators": f"{ANN_NAMES[a]} vs. {ANN_NAMES[b]}", "Construct": lab, "Posts": j.post_id.nunique(), "Pairs": len(j),
                     "Same winner": pct(r["same"]), "Cohen's kappa (winner)": f2(cohen_kappa(np.sign(j[x]), np.sign(j[y]))),
                     "Exact 4-point label": f"{100*np.mean(j[x] == j[y]):.1f}%", "Weighted kappa": ci(r["wk"]),
                     "Krippendorff's alpha (ordinal)": ci(r["alpha"])})
A2 = pd.DataFrame(rows)
save(A2, "A2_human_agreement", "Agreement between human annotators on the same pairs", "95% CIs resample posts.")

WH = human_rankings(H, responses)
rows = []; RELP = {}
for a, b in (("H1", "H3"), ("H1", "H2"), ("H3", "H2")):
    for t, (lab, _) in CONSTRUCTS.items():
        d = WH[[ "post_id", f"{a}_{t}", f"{b}_{t}"]].dropna()
        tau = kendall_tau_b_by_op(d, f"{a}_{t}", f"{b}_{t}").mean()
        rho = d[[f"{a}_{t}", f"{b}_{t}"]].rank().corr().iloc[0, 1]
        RELP[(a, b, t)] = rho
        rows.append({"Annotators": f"{ANN_NAMES[a]} vs. {ANN_NAMES[b]}", "Construct": lab, "Posts": d.post_id.nunique(),
                     "Ranking agreement (tau-b, per post)": f2(tau), "Rank r = one-annotator reliability": f2(rho)})
rel_cal = human_reliability(CAL, LAM)
REL_H = {t: np.mean([RELP[(a, b, t)] for a, b in (("H1", "H3"), ("H1", "H2"), ("H3", "H2"))]) for t in CONSTRUCTS}
for t, (lab, _) in CONSTRUCTS.items():
    rows.append({"Annotators": "Mean of the three pairs (used below)", "Construct": lab, "Posts": len(H3_OPS),
                 "Ranking agreement (tau-b, per post)": "--", "Rank r = one-annotator reliability": f2(REL_H[t])})
    if rel_cal:
        rows.append({"Annotators": f"Calibration ({len(rel_cal['annotators'])} annotators)", "Construct": lab, "Posts": rel_cal["ops"],
                     "Ranking agreement (tau-b, per post)": f2(rel_cal[t]["tau_pairwise_mean"]),
                     "Rank r = one-annotator reliability": f2(rel_cal[t]["rho_pairwise_mean"])})
A3 = pd.DataFrame(rows)
save(A3, "A3_human_reliability", "Reliability of one human annotator's within-post ranking",
     "Rank r between two annotators' within-post percentiles, pooled over posts. The mean of the three pairs is used to correct the human associations for unreliability.")

**A1_annotators. How each human annotator used the scale**  
_Different C and E winners: share of pairs where the cogency and effectiveness choices pick different responses._

,Annotator,Posts,Pairs,Different C and E winners,Identical C and E label,Strong (C / E),Median seconds per pair
0,H1,115,1725,12.2%,72.6%,62% / 60%,99
1,H2,115,1725,0.5%,90.3%,79% / 75%,17
2,H3,58,870,8.3%,75.5%,63% / 69%,55


**A2_human_agreement. Agreement between human annotators on the same pairs**  
_95% CIs resample posts._

,Annotators,Construct,Posts,Pairs,Same winner,Cohen's kappa (winner),Exact 4-point label,Weighted kappa,Krippendorff's alpha (ordinal)
0,H1 vs. H3,cogency,58,870,"78.5% [74.5, 82.1]",0.57,55.4%,"0.57 [0.50, 0.62]","0.70 [0.63, 0.75]"
1,H1 vs. H3,effectiveness,58,870,"77.8% [74.0, 81.5]",0.55,53.4%,"0.53 [0.47, 0.59]","0.66 [0.59, 0.72]"
2,H1 vs. H2,cogency,115,1725,"78.4% [75.8, 81.1]",0.57,52.9%,"0.51 [0.46, 0.55]","0.61 [0.56, 0.66]"
3,H1 vs. H2,effectiveness,115,1725,"72.9% [70.2, 75.5]",0.46,49.9%,"0.44 [0.40, 0.48]","0.53 [0.49, 0.58]"
4,H3 vs. H2,cogency,58,870,"76.0% [71.4, 80.3]",0.52,51.1%,"0.47 [0.39, 0.55]","0.57 [0.47, 0.65]"
5,H3 vs. H2,effectiveness,58,870,"76.4% [72.2, 80.2]",0.53,53.0%,"0.50 [0.42, 0.56]","0.60 [0.52, 0.68]"


**A3_human_reliability. Reliability of one human annotator's within-post ranking**  
_Rank r between two annotators' within-post percentiles, pooled over posts. The mean of the three pairs is used to correct the human associations for unreliability._

,Annotators,Construct,Posts,"Ranking agreement (tau-b, per post)",Rank r = one-annotator reliability
0,H1 vs. H3,cogency,58,0.62,0.71
1,H1 vs. H3,effectiveness,58,0.59,0.69
2,H1 vs. H2,cogency,115,0.59,0.69
3,H1 vs. H2,effectiveness,115,0.48,0.59
4,H3 vs. H2,cogency,58,0.52,0.60
5,H3 vs. H2,effectiveness,58,0.54,0.63
6,Mean of the three pairs (used below),cogency,58,--,0.67
7,Calibration (3 annotators),cogency,5,0.61,0.72
8,Mean of the three pairs (used below),effectiveness,58,--,0.64
9,Calibration (3 annotators),effectiveness,5,0.51,0.62


## Part B. Study 1 with H1 as the human reference (115 posts)

Same estimators as the thesis: pairwise agreement with the human, agreement by the human's preference strength, order consistency, the multitrait–multimethod matrix and contrasts (ΔC, ΔE, Γ; Holm over six), and the audience-specificity increment Λ.

In [5]:
HS = H["H1"]
J  = judgments(HS, llm)
W  = wide(scores(J, responses, LAM))
W_strength = wide(scores(J, responses, LAM, "bt_strength"))
W_signed   = wide(scores(J, responses, LAM, "signed"))
SUB_OPS = set(HS.post_id)
rel_llm = split_half_reliability(J[J.post_id.isin(SUB_OPS)], responses, LAM)
REL = {("H", t): REL_H[t] for t in CONSTRUCTS}
REL.update({(mod, t): r["rho_full_sb"] for (mod, t), r in rel_llm.items()})

rows = []
for (mod, t), r in rel_llm.items():
    rows.append({"Method": METHOD_NAMES[mod], "Construct": CONSTRUCTS[t][0], "Reliability (Spearman-Brown)": f2(r["rho_full_sb"]),
                 "Basis": "A/B vs. B/A order, 115 posts"})
for t in CONSTRUCTS:
    rows.append({"Method": "Human (one annotator)", "Construct": CONSTRUCTS[t][0], "Reliability (Spearman-Brown)": f2(REL_H[t]),
                 "Basis": "mean of 3 annotator pairs (Table A3)"})
save(pd.DataFrame(rows), "B0_reliability", "Reliability of the within-post rankings")

# ---- B1 pairwise agreement with the human (each model judgment vs. the human label)
HL = human_long(HS)
PJ = LL.merge(HL, on=["construct", "pair_key", "post_id"])
rows = []; B1 = {}
for mod in ("gpt", "claude"):
    for t, (lab, _) in CONSTRUCTS.items():
        d = PJ[(PJ.model == mod) & (PJ.construct == t)]
        st = lambda d: {"same": np.mean(np.sign(d.s) == np.sign(d.h)), "exact": np.mean(d.s == d.h),
                        "wk": weighted_kappa(d.s.values, d.h.values)}
        r = boot_df(d, st, B_PAIRS, seed=2); B1[(mod, t)] = r
        rows.append({"Model": METHOD_NAMES[mod], "Construct": lab, "Judgments": len(d), "Same winner": pct(r["same"]),
                     "Kappa (winner)": f2(cohen_kappa(np.sign(d.s), np.sign(d.h))), "Exact 4-point label": pct(r["exact"]),
                     "Weighted kappa": ci(r["wk"])})
save(pd.DataFrame(rows), "B1_pairwise", "LLM agreement with the human (H1) on individual pairs",
     "Each pair is judged twice by each model (both presentation orders). 95% CIs resample posts.")

# ---- B2 agreement by the human's preference strength
rows = []; B2 = {}
for mod in ("gpt", "claude"):
    for t, (lab, _) in CONSTRUCTS.items():
        d = PJ[(PJ.model == mod) & (PJ.construct == t)]
        for nm, sel in (("Strong", d.h.abs() == 2), ("Weak", d.h.abs() == 1)):
            r = boot_df(d[sel], lambda d: {"same": np.mean(np.sign(d.s) == np.sign(d.h))}, B_PAIRS, seed=3)
            B2[(mod, t, nm)] = r["same"]
            rows.append({"Model": METHOD_NAMES[mod], "Construct": lab, "Human said": nm, "Pairs": int(sel.sum() / 2),
                         "Same winner": pct(r["same"]), "Model says Strong": f"{100*np.mean(d[sel].s.abs() == 2):.1f}%"})
for t, (lab, pre) in CONSTRUCTS.items():
    rows.append({"Model": "Human (H1)", "Construct": lab, "Human said": "all", "Pairs": len(HS), "Same winner": "--",
                 "Model says Strong": f"{100*np.mean(HS[f'{pre}_signed_canon'].abs() == 2):.1f}% (human)"})
save(pd.DataFrame(rows), "B2_strength", "Agreement with the human by the human's preference strength")

# ---- B3 presentation-order consistency
rows = []; B3 = {}
for mod in ("gpt", "claude"):
    for t, (lab, _) in CONSTRUCTS.items():
        d = LL[(LL.model == mod) & (LL.construct == t) & LL.post_id.isin(SUB_OPS)]
        pv = d.pivot_table(index=["pair_key", "post_id"], columns="orientation", values="s").dropna().reset_index()
        pv["flip"] = np.sign(pv.AB) != np.sign(pv.BA)
        pv["weak_both"] = (pv.AB.abs() == 1) & (pv.BA.abs() == 1)
        pv = pv.merge(HL[HL.construct == t][["pair_key", "h"]], on="pair_key")
        first = np.mean(np.where(d.s < 0, d.lo, d.hi) == d.shown_first)
        r = boot_df(pv, lambda d: {"cons": 1 - d.flip.mean(), "fs": d[d.h.abs() == 2].flip.mean(), "fw": d[d.h.abs() == 1].flip.mean()}, B_PAIRS, seed=4)
        B3[(mod, t)] = {"cons": r["cons"], "share_weak": pv[pv.flip].weak_both.mean(), "flips": int(pv.flip.sum())}
        rows.append({"Model": METHOD_NAMES[mod], "Construct": lab, "Pairs": len(pv), "Same winner in both orders": pct(r["cons"]),
                     "Flipped pairs": int(pv.flip.sum()), "Picked first-shown": f"{100*first:.1f}%",
                     "Flip rate, human Strong": pct(r["fs"]), "Flip rate, human Weak": pct(r["fw"]),
                     "Flips on pairs the model rated Weak in both orders": f"{100*pv[pv.flip].weak_both.mean():.1f}%"})
save(pd.DataFrame(rows), "B3_order", "Consistency of each model across the two presentation orders")

**B0_reliability. Reliability of the within-post rankings**

,Method,Construct,Reliability (Spearman-Brown),Basis
0,GPT-6.1 Sol,cogency,0.98,"A/B vs. B/A order, 115 posts"
1,GPT-6.1 Sol,effectiveness,0.97,"A/B vs. B/A order, 115 posts"
2,Claude Sonnet 5.5,cogency,0.95,"A/B vs. B/A order, 115 posts"
3,Claude Sonnet 5.5,effectiveness,0.95,"A/B vs. B/A order, 115 posts"
4,Human (one annotator),cogency,0.67,mean of 3 annotator pairs (Table A3)
5,Human (one annotator),effectiveness,0.64,mean of 3 annotator pairs (Table A3)


**B1_pairwise. LLM agreement with the human (H1) on individual pairs**  
_Each pair is judged twice by each model (both presentation orders). 95% CIs resample posts._

,Model,Construct,Judgments,Same winner,Kappa (winner),Exact 4-point label,Weighted kappa
0,GPT-6.1 Sol,cogency,3450,"81.4% [79.4, 83.3]",0.63,"58.2% [55.5, 60.8]","0.61 [0.57, 0.64]"
1,GPT-6.1 Sol,effectiveness,3450,"81.9% [79.7, 83.9]",0.64,"59.3% [56.6, 62.0]","0.62 [0.59, 0.65]"
2,Claude Sonnet 5.5,cogency,3450,"85.8% [84.3, 87.3]",0.72,"60.2% [57.3, 63.1]","0.65 [0.62, 0.68]"
3,Claude Sonnet 5.5,effectiveness,3450,"80.1% [78.1, 81.9]",0.60,"52.6% [49.8, 55.5]","0.56 [0.53, 0.59]"


**B2_strength. Agreement with the human by the human's preference strength**

,Model,Construct,Human said,Pairs,Same winner,Model says Strong
0,GPT-6.1 Sol,cogency,Strong,1067,"88.4% [85.7, 91.0]",66.9%
1,GPT-6.1 Sol,cogency,Weak,658,"70.0% [66.8, 73.0]",27.1%
2,GPT-6.1 Sol,effectiveness,Strong,1028,"90.9% [88.3, 93.5]",72.3%
3,GPT-6.1 Sol,effectiveness,Weak,697,"68.5% [64.9, 72.1]",31.4%
4,Claude Sonnet 5.5,cogency,Strong,1067,"93.8% [92.3, 95.3]",63.7%
5,Claude Sonnet 5.5,cogency,Weak,658,"72.8% [69.5, 75.9]",19.5%
6,Claude Sonnet 5.5,effectiveness,Strong,1028,"90.5% [88.2, 92.7]",54.5%
7,Claude Sonnet 5.5,effectiveness,Weak,697,"64.7% [61.1, 67.9]",18.1%
8,Human (H1),cogency,all,1725,--,61.9% (human)
9,Human (H1),effectiveness,all,1725,--,59.6% (human)


**B3_order. Consistency of each model across the two presentation orders**

,Model,Construct,Pairs,Same winner in both orders,Flipped pairs,Picked first-shown,"Flip rate, human Strong","Flip rate, human Weak",Flips on pairs the model rated Weak in both orders
0,GPT-6.1 Sol,cogency,1725,"94.7% [93.5, 95.8]",92,48.7%,"3.5% [2.4, 4.7]","8.4% [6.4, 10.4]",100.0%
1,GPT-6.1 Sol,effectiveness,1725,"94.4% [93.4, 95.4]",96,49.1%,"2.6% [1.7, 3.7]","9.9% [7.9, 11.9]",99.0%
2,Claude Sonnet 5.5,cogency,1725,"91.4% [90.0, 92.8]",148,52.1%,"5.1% [3.7, 6.4]","14.3% [11.4, 17.2]",97.3%
3,Claude Sonnet 5.5,effectiveness,1725,"90.1% [88.6, 91.6]",170,50.5%,"6.0% [4.5, 7.8]","15.5% [13.2, 17.7]",97.1%


In [6]:
COLS = ["H_C", "H_E", "gpt_C", "gpt_E", "claude_C", "claude_E"]
ids, M = op_matrices(W[W.post_id.isin(SUB_OPS)], COLS)
print(f"{len(ids)} posts with complete rankings")
R1 = {kind: boot_matrix_stats(M, lambda M, k=kind: {**study1_stats(M, k), **lambda_stats(M)}, B) for kind in ("tau", "rho")}

lab = {"C": "cogency", "E": "effectiveness"}
rows = []
for mod in ("gpt", "claude"):
    for t in "CE":
        row = {"LLM measurement": f"{METHOD_NAMES[mod]} {lab[t]}"}
        for ht in "CE":
            row[f"Human {lab[ht]} (tau-b)"] = ci(R1["tau"][f"A|H_{ht}|{mod}_{t}"])
            row[f"Human {lab[ht]} (rank r)"] = ci(R1["rho"][f"A|H_{ht}|{mod}_{t}"])
        rows.append(row)
save(pd.DataFrame(rows), "S1_associations", "Associations between LLM and human (H1) rankings",
     "Mean within-post Kendall tau-b and pooled rank correlation, 95% post-bootstrap CIs. Same-construct cells are the diagonal.")

rows = []
for t in "CE":
    rows.append({"Construct": lab[t], "GPT vs. Claude (tau-b)": ci(R1["tau"][f"A|gpt_{t}|claude_{t}"]),
                 "Human vs. GPT (tau-b)": ci(R1["tau"][f"A|H_{t}|gpt_{t}"]), "Human vs. Claude (tau-b)": ci(R1["tau"][f"A|H_{t}|claude_{t}"])})
save(pd.DataFrame(rows), "S1_model_model", "Model-model vs. human-model agreement on the same construct")

keys = ["Delta_C|gpt", "Delta_E|gpt", "Gamma|gpt", "Delta_C|claude", "Delta_E|claude", "Gamma|claude"]
ADJ = dict(zip(keys, holm([R1["tau"][k][3] for k in keys])))
rows = []
for k in keys:
    q, mod = k.split("|")
    rows.append({"Contrast": {"Delta_C": "Delta C (cogency)", "Delta_E": "Delta E (effectiveness)", "Gamma": "Gamma (coupling)"}[q],
                 "Model": METHOD_NAMES[mod], "Estimate, tau-b [95% CI]": ci(R1["tau"][k]), "Holm-adjusted p": pval(ADJ[k]),
                 "Estimate, rank r [95% CI]": ci(R1["rho"][k])})
save(pd.DataFrame(rows), "S1_contrasts", "Selective correspondence (Delta) and construct coupling (Gamma), human = H1",
     "Holm adjustment over the six contrasts. Human coupling A(human C, human E) = " + ci(R1["tau"]["A|H_C|H_E"]) + " (tau-b).")

rows = []; LAMBDA = {}
for mod in ("H", "gpt", "claude"):
    e, lo, hi, _ = R1["rho"][f"Lambda|{mod}"]
    r = pooled_r(M[f"{mod}_C"], M[f"{mod}_E"])
    relC, relE = REL[(mod, "C")], REL[(mod, "E")]
    rc = r / np.sqrt(relC * relE)
    LAMBDA[mod] = (r, e, lo, hi, relC, relE, rc)
    rows.append({"Method": METHOD_NAMES[mod] if mod != "H" else "Human (H1)", "Correlation C-E (rank r)": f2(r),
                 "Lambda observed [95% CI]": ci((e, lo, hi)), "Reliability C / E": f"{f2(relC)} / {f2(relE)}",
                 "Correlation corrected for unreliability": f2(min(rc, 1.0)),
                 "Lambda corrected": f2(max(0.0, 1 - min(rc, 1.0) ** 2))})
save(pd.DataFrame(rows), "S1_lambda", "Audience-specificity increment: share of the effectiveness ranking not explained by cogency",
     "Lambda = 1 - r^2. Correction divides r by the square root of the two reliabilities; values at the boundary are capped at 1.")

rows = []
for vname, Wv in (("Bradley-Terry, winners only (primary)", W), ("Bradley-Terry, Strong counted twice", W_strength),
                  ("Sum of signed labels (no model)", W_signed)):
    _, Mv = op_matrices(Wv[Wv.post_id.isin(SUB_OPS)], COLS)
    r = boot_matrix_stats(Mv, lambda M: study1_stats(M, "tau"), B)
    for mod in ("gpt", "claude"):
        rows.append({"Scoring": vname, "Model": METHOD_NAMES[mod], "Delta C": ci(r[f"Delta_C|{mod}"]),
                     "Delta E": ci(r[f"Delta_E|{mod}"]), "Gamma": ci(r[f"Gamma|{mod}"])})
save(pd.DataFrame(rows), "S1_robustness", "Study 1 contrasts under alternative scoring of the pairwise judgments")

115 posts with complete rankings


**S1_associations. Associations between LLM and human (H1) rankings**  
_Mean within-post Kendall tau-b and pooled rank correlation, 95% post-bootstrap CIs. Same-construct cells are the diagonal._

,LLM measurement,Human cogency (tau-b),Human cogency (rank r),Human effectiveness (tau-b),Human effectiveness (rank r)
0,GPT-6.1 Sol cogency,"0.66 [0.62, 0.70]","0.76 [0.73, 0.80]","0.63 [0.59, 0.68]","0.74 [0.70, 0.78]"
1,GPT-6.1 Sol effectiveness,"0.69 [0.65, 0.74]","0.78 [0.75, 0.82]","0.67 [0.62, 0.71]","0.76 [0.72, 0.80]"
2,Claude Sonnet 5.5 cogency,"0.76 [0.73, 0.79]","0.86 [0.83, 0.88]","0.61 [0.57, 0.65]","0.72 [0.68, 0.76]"
3,Claude Sonnet 5.5 effectiveness,"0.73 [0.69, 0.76]","0.82 [0.79, 0.85]","0.64 [0.60, 0.68]","0.75 [0.71, 0.79]"


**S1_model_model. Model-model vs. human-model agreement on the same construct**

,Construct,GPT vs. Claude (tau-b),Human vs. GPT (tau-b),Human vs. Claude (tau-b)
0,cogency,"0.64 [0.60, 0.68]","0.66 [0.62, 0.70]","0.76 [0.73, 0.79]"
1,effectiveness,"0.69 [0.65, 0.73]","0.67 [0.62, 0.71]","0.64 [0.60, 0.68]"


**S1_contrasts. Selective correspondence (Delta) and construct coupling (Gamma), human = H1**  
_Holm adjustment over the six contrasts. Human coupling A(human C, human E) = 0.76 [0.72, 0.81] (tau-b)._

,Contrast,Model,"Estimate, tau-b [95% CI]",Holm-adjusted p,"Estimate, rank r [95% CI]"
0,Delta C (cogency),GPT-6.1 Sol,"-0.03 [-0.07, 0.01]",0.316,"-0.02 [-0.06, 0.02]"
1,Delta E (effectiveness),GPT-6.1 Sol,"0.03 [0.00, 0.07]",0.225,"0.03 [-0.01, 0.06]"
2,Gamma (coupling),GPT-6.1 Sol,"-0.00 [-0.06, 0.05]",1.000,"-0.00 [-0.05, 0.04]"
3,Delta C (cogency),Claude Sonnet 5.5,"0.04 [0.00, 0.07]",0.198,"0.04 [0.01, 0.07]"
4,Delta E (effectiveness),Claude Sonnet 5.5,"0.03 [-0.00, 0.07]",0.316,"0.03 [-0.00, 0.06]"
5,Gamma (coupling),Claude Sonnet 5.5,"0.01 [-0.05, 0.07]",1.000,"0.01 [-0.05, 0.06]"


**S1_lambda. Audience-specificity increment: share of the effectiveness ranking not explained by cogency**  
_Lambda = 1 - r^2. Correction divides r by the square root of the two reliabilities; values at the boundary are capped at 1._

,Method,Correlation C-E (rank r),Lambda observed [95% CI],Reliability C / E,Correlation corrected for unreliability,Lambda corrected
0,Human (H1),0.84,"0.29 [0.23, 0.35]",0.67 / 0.64,1.00,0.00
1,GPT-6.1 Sol,0.84,"0.29 [0.24, 0.35]",0.98 / 0.97,0.86,0.26
2,Claude Sonnet 5.5,0.85,"0.28 [0.22, 0.34]",0.95 / 0.95,0.89,0.20


**S1_robustness. Study 1 contrasts under alternative scoring of the pairwise judgments**

,Scoring,Model,Delta C,Delta E,Gamma
0,"Bradley-Terry, winners only (primary)",GPT-6.1 Sol,"-0.03 [-0.07, 0.01]","0.03 [0.00, 0.07]","-0.00 [-0.06, 0.05]"
1,"Bradley-Terry, winners only (primary)",Claude Sonnet 5.5,"0.04 [0.00, 0.07]","0.03 [-0.00, 0.07]","0.01 [-0.05, 0.07]"
2,"Bradley-Terry, Strong counted twice",GPT-6.1 Sol,"-0.03 [-0.07, 0.01]","0.03 [-0.00, 0.06]","-0.01 [-0.06, 0.04]"
3,"Bradley-Terry, Strong counted twice",Claude Sonnet 5.5,"0.03 [-0.00, 0.06]","0.03 [-0.01, 0.07]","0.01 [-0.05, 0.07]"
4,Sum of signed labels (no model),GPT-6.1 Sol,"-0.02 [-0.06, 0.02]","0.03 [-0.00, 0.06]","-0.02 [-0.07, 0.03]"
5,Sum of signed labels (no model),Claude Sonnet 5.5,"0.03 [-0.01, 0.06]","0.04 [0.00, 0.07]","0.00 [-0.05, 0.06]"


## Part C. Study 1 with each human as the reference

The same Study 1 quantities with H3 (58 posts) and H2 (115 posts) as the human reference, and with H1 restricted to H3's 58 posts (so the two can be compared on the same posts).

In [7]:
def study1_for(hs, ops, seed=0):
    Jr = judgments(hs, llm); Wr = wide(scores(Jr, responses, LAM))
    _, Mr = op_matrices(Wr[Wr.post_id.isin(ops)], COLS)
    r = boot_matrix_stats(Mr, lambda M: study1_stats(M, "tau"), B, seed)
    adj = dict(zip(keys, holm([r[k][3] for k in keys])))
    return r, adj, Jr

REFS = {"H1, 115 posts": (H["H1"], MAIN_OPS), "H1, H3's 58 posts": (H["H1"][H["H1"].post_id.isin(H3_OPS)], H3_OPS),
        "H3, 58 posts": (H["H3"], H3_OPS), "H2, 115 posts": (H["H2"], MAIN_OPS)}
BYREF = {}; rows = []; JREF = {}
for name, (hs, ops) in REFS.items():
    r, adj, Jr = study1_for(hs, ops)
    BYREF[name] = (r, adj); JREF[name] = Jr
    for mod in ("gpt", "claude"):
        rows.append({"Human reference": name, "Model": METHOD_NAMES[mod],
                     "Matching tau-b, C / E": f"{f2(r[f'A|H_C|{mod}_C'][0])} / {f2(r[f'A|H_E|{mod}_E'][0])}",
                     "Delta C [CI], Holm p": f"{ci(r[f'Delta_C|{mod}'])}, {pval(adj[f'Delta_C|{mod}'])}",
                     "Delta E [CI], Holm p": f"{ci(r[f'Delta_E|{mod}'])}, {pval(adj[f'Delta_E|{mod}'])}",
                     "Gamma [CI], Holm p": f"{ci(r[f'Gamma|{mod}'])}, {pval(adj[f'Gamma|{mod}'])}",
                     "Human coupling tau-b": f2(r["A|H_C|H_E"][0])})
save(pd.DataFrame(rows), "C1_study1_by_reference", "Study 1 with each human annotator as the reference",
     "Mean within-post Kendall tau-b; 95% post-bootstrap CIs; Holm over the six contrasts within each reference.")

rows = []
for name, (hs, ops) in REFS.items():
    hl = human_long(hs); pj = LL[LL.post_id.isin(ops)].merge(hl, on=["construct", "pair_key", "post_id"])
    for mod in ("gpt", "claude"):
        row = {"Human reference": name, "Model": METHOD_NAMES[mod]}
        for t, (lab_, _) in CONSTRUCTS.items():
            d = pj[(pj.model == mod) & (pj.construct == t)]
            row[f"Same winner, {lab_}"] = f"{100*np.mean(np.sign(d.s) == np.sign(d.h)):.1f}%"
        rows.append(row)
save(pd.DataFrame(rows), "C2_pairwise_by_reference", "LLM same-winner agreement with each human annotator")

**C1_study1_by_reference. Study 1 with each human annotator as the reference**  
_Mean within-post Kendall tau-b; 95% post-bootstrap CIs; Holm over the six contrasts within each reference._

,Human reference,Model,"Matching tau-b, C / E","Delta C [CI], Holm p","Delta E [CI], Holm p","Gamma [CI], Holm p",Human coupling tau-b
0,"H1, 115 posts",GPT-6.1 Sol,0.66 / 0.67,"-0.03 [-0.07, 0.01], 0.316","0.03 [0.00, 0.07], 0.225","-0.00 [-0.06, 0.05], 1.000",0.76
1,"H1, 115 posts",Claude Sonnet 5.5,0.76 / 0.64,"0.04 [0.00, 0.07], 0.198","0.03 [-0.00, 0.07], 0.316","0.01 [-0.05, 0.07], 1.000",0.76
2,"H1, H3's 58 posts",GPT-6.1 Sol,0.65 / 0.68,"-0.04 [-0.09, 0.01], 0.364","0.06 [0.01, 0.11], 0.075","-0.02 [-0.11, 0.06], 1.000",0.77
3,"H1, H3's 58 posts",Claude Sonnet 5.5,0.77 / 0.64,"0.03 [-0.01, 0.07], 0.435","0.05 [0.01, 0.09], 0.054","0.02 [-0.06, 0.10], 1.000",0.77
4,"H3, 58 posts",GPT-6.1 Sol,0.51 / 0.59,"-0.02 [-0.08, 0.04], 0.966","0.03 [-0.02, 0.08], 0.788","-0.10 [-0.18, -0.02], 0.114",0.84
5,"H3, 58 posts",Claude Sonnet 5.5,0.61 / 0.62,"0.04 [-0.02, 0.11], 0.788","-0.00 [-0.05, 0.04], 0.966","-0.06 [-0.12, 0.02], 0.745",0.84
6,"H2, 115 posts",GPT-6.1 Sol,0.43 / 0.44,"-0.01 [-0.05, 0.03], 1.000","0.00 [-0.03, 0.05], 1.000","-0.23 [-0.27, -0.19], 0.006",0.99
7,"H2, 115 posts",Claude Sonnet 5.5,0.57 / 0.52,"0.05 [0.01, 0.09], 0.052","-0.05 [-0.08, -0.01], 0.052","-0.21 [-0.25, -0.18], 0.006",0.99


**C2_pairwise_by_reference. LLM same-winner agreement with each human annotator**

,Human reference,Model,"Same winner, cogency","Same winner, effectiveness"
0,"H1, 115 posts",GPT-6.1 Sol,81.4%,81.9%
1,"H1, 115 posts",Claude Sonnet 5.5,85.8%,80.1%
2,"H1, H3's 58 posts",GPT-6.1 Sol,80.9%,82.5%
3,"H1, H3's 58 posts",Claude Sonnet 5.5,86.1%,80.0%
4,"H3, 58 posts",GPT-6.1 Sol,73.7%,78.1%
5,"H3, 58 posts",Claude Sonnet 5.5,78.5%,79.0%
6,"H2, 115 posts",GPT-6.1 Sol,70.6%,70.5%
7,"H2, 115 posts",Claude Sonnet 5.5,77.2%,74.6%


## Part C2. Additional diagnostics reported in the thesis

Order effects, preference strength, model consensus, construct gaps and a flagged-response check. These use only the labels (no outcomes). Every number in the thesis that is not in an earlier table comes from here.

In [8]:
# ---- X1 share of Strong labels (all judgments) and X3 first-shown preference
rows = []
for mod in ("claude", "gpt"):
    for t, (lab_, _) in CONSTRUCTS.items():
        d = LL[(LL.model == mod) & (LL.construct == t)].copy()
        d["first"] = np.where(d.s < 0, d.lo, d.hi) == d.shown_first
        r = boot_df(d, lambda d: {"f": d["first"].mean()}, B_PAIRS, seed=11)["f"]
        rows.append({"Model": METHOD_NAMES[mod], "Construct": lab_, "Judgments": len(d),
                     "Share labeled Strong": f"{100*np.mean(d.s.abs() == 2):.1f}%", "Picked the response shown first": pct(r)})
save(pd.DataFrame(rows), "X1_strength_and_position", "Share of Strong labels and preference for the response shown first (115 posts)")

# ---- X2 paired Claude - GPT agreement with the human, and strong disagreements
rows = []
for t, (lab_, _) in CONSTRUCTS.items():
    a = PJ[PJ.construct == t].pivot_table(index=["pair_key", "post_id", "orientation", "h"], columns="model", values="s").reset_index()
    st = lambda d: {"d_same": np.mean(np.sign(d.claude) == np.sign(d.h)) - np.mean(np.sign(d.gpt) == np.sign(d.h)),
                    "d_wk": weighted_kappa(d.claude.values, d.h.values) - weighted_kappa(d.gpt.values, d.h.values)}
    r = boot_df(a, st, B_PAIRS, seed=5)
    sd = {m: int(((a[m].abs() == 2) & (a.h.abs() == 2) & (np.sign(a[m]) != np.sign(a.h))).sum()) for m in ("claude", "gpt")}
    rows.append({"Construct": lab_, "Claude - GPT, same winner (points)": f"{100*r['d_same'][0]:+.1f} [{100*r['d_same'][1]:+.1f}, {100*r['d_same'][2]:+.1f}]",
                 "Claude - GPT, weighted kappa": f"{r['d_wk'][0]:+.3f} [{r['d_wk'][1]:+.3f}, {r['d_wk'][2]:+.3f}]",
                 "Strong disagreements, Claude": sd["claude"], "Strong disagreements, GPT": sd["gpt"]})
save(pd.DataFrame(rows), "X2_paired_models", "Paired comparison of the two models' agreement with the human (same human labels)",
     "Strong disagreement: model and human both say Strong for opposite responses (judgment level).")

# ---- X4 flips by each model's own strength (Appendix table)
rows = []
for mod in ("claude", "gpt"):
    for t, (lab_, _) in CONSTRUCTS.items():
        d = LL[(LL.model == mod) & (LL.construct == t)]
        pv = d.pivot_table(index=["pair_key", "post_id"], columns="orientation", values="s").dropna().reset_index()
        pv["flip"] = np.sign(pv.AB) != np.sign(pv.BA)
        pv["cat"] = np.where((pv.AB.abs() == 2) & (pv.BA.abs() == 2), "Strong in both",
                    np.where((pv.AB.abs() == 1) & (pv.BA.abs() == 1), "Weak in both", "Strong in one"))
        for c in ("Strong in both", "Strong in one", "Weak in both"):
            s_ = pv[pv.cat == c]
            r = boot_df(s_, lambda d: {"f": d.flip.mean()}, B_PAIRS, seed=9)["f"] if s_.flip.sum() > 0 else (0.0, 0.0, 0.0)
            rows.append({"Model": METHOD_NAMES[mod], "Construct": lab_, "Strength in the two orders": c, "Pairs": len(s_),
                         "Flip rate": pct(r), "Share of all flips": f"{100*s_.flip.sum()/pv.flip.sum():.1f}%"})
save(pd.DataFrame(rows), "X4_flips_by_model_strength", "Order-dependent flips by the strength of each model's own labels (115 posts)")

# ---- X5 consensus of the four model judgments vs. the human
rows = []
for t, (lab_, _) in CONSTRUCTS.items():
    d = PJ[PJ.construct == t]
    g_ = d.groupby(["pair_key", "post_id"]).agg(n=("s", "size"), votes=("s", lambda s: np.sign(s).sum()), h=("h", "first")).reset_index()
    un, sp, ti = g_[g_.votes.abs() == 4], g_[g_.votes.abs() == 2], g_[g_.votes == 0]
    ru = boot_df(un, lambda d: {"a": np.mean(np.sign(d.votes) == np.sign(d.h))}, B_PAIRS, seed=7)["a"]
    rs = boot_df(sp, lambda d: {"a": np.mean(np.sign(d.votes) == np.sign(d.h))}, B_PAIRS, seed=8)["a"]
    du, ds = int((np.sign(un.votes) != np.sign(un.h)).sum()), int((np.sign(sp.votes) != np.sign(sp.h)).sum())
    rows.append({"Construct": lab_, "Pairs": len(g_), "All four agree": f"{len(un)} ({100*len(un)/len(g_):.1f}%)",
                 "Match human when all agree": pct(ru), "Split 3 to 1": len(sp), "Majority matches human when split": pct(rs),
                 "Tied 2 to 2": len(ti), "Human disagreements that were unanimous": f"{du} of {du+ds} ({100*du/(du+ds):.1f}%)"})
save(pd.DataFrame(rows), "X5_model_consensus", "Agreement of the four model judgments (two models, two orders) with the human",
     "Last column: among pairs whose clear model majority disagrees with the human, the share on which all four judgments agreed.")

# ---- X6 construct gap |G| per response
rows = []
for nm, col in (("Human annotation (H1)", "H"), ("Claude Sonnet 5.5", "claude"), ("GPT-6.1 Sol", "gpt")):
    G_ = (W[f"{col}_E"] - W[f"{col}_C"]).dropna()
    rows.append({"Method": nm, "Responses": len(G_), "Mean |G|": f3(G_.abs().mean()), "|G| >= 1/3": f"{100*(G_.abs() >= 1/3 - 1e-9).mean():.1f}%"})
for nm, a in (("H2", "H2"), ("H3", "H3")):
    G_ = (WH[f"{a}_E"] - WH[f"{a}_C"]).dropna()
    rows.append({"Method": nm, "Responses": len(G_), "Mean |G|": f3(G_.abs().mean()), "|G| >= 1/3": f"{100*(G_.abs() >= 1/3 - 1e-9).mean():.1f}%"})
save(pd.DataFrame(rows), "X6_construct_gap", "Construct gap G = effectiveness percentile - cogency percentile, per response")

# ---- X7 agreement excluding pairs with a response flagged by the human annotator
fl = HS[["a_nonargumentative", "b_nonargumentative", "a_unreadable", "b_unreadable"]].astype(str).apply(lambda c: c.str.lower().isin(["true", "1"])).any(axis=1)
bad = set(HS.pair_key[fl]); rows = []
for mod in ("claude", "gpt"):
    for t, (lab_, _) in CONSTRUCTS.items():
        d = PJ[(PJ.model == mod) & (PJ.construct == t)]; k = d[~d.pair_key.isin(bad)]
        rows.append({"Model": METHOD_NAMES[mod], "Construct": lab_, "Same winner, all pairs": f"{100*np.mean(np.sign(d.s) == np.sign(d.h)):.1f}%",
                     "Same winner, flagged pairs excluded": f"{100*np.mean(np.sign(k.s) == np.sign(k.h)):.1f}%", "Flagged pairs": len(bad)})
save(pd.DataFrame(rows), "X7_flagged_excluded", "Agreement with the human excluding pairs with a response flagged as non-argumentative or unreadable")

**X1_strength_and_position. Share of Strong labels and preference for the response shown first (115 posts)**

,Model,Construct,Judgments,Share labeled Strong,Picked the response shown first
0,Claude Sonnet 5.5,cogency,3450,46.8%,"52.1% [51.4, 52.9]"
1,Claude Sonnet 5.5,effectiveness,3450,39.8%,"50.5% [49.7, 51.4]"
2,GPT-6.1 Sol,cogency,3450,51.7%,"48.7% [48.1, 49.3]"
3,GPT-6.1 Sol,effectiveness,3450,55.8%,"49.1% [48.5, 49.7]"


**X2_paired_models. Paired comparison of the two models' agreement with the human (same human labels)**  
_Strong disagreement: model and human both say Strong for opposite responses (judgment level)._

,Construct,"Claude - GPT, same winner (points)","Claude - GPT, weighted kappa","Strong disagreements, Claude","Strong disagreements, GPT"
0,cogency,"+4.4 [+2.1, +6.7]","+0.045 [+0.014, +0.074]",13,34
1,effectiveness,"-1.8 [-4.0, +0.4]","-0.062 [-0.094, -0.030]",18,47


**X4_flips_by_model_strength. Order-dependent flips by the strength of each model's own labels (115 posts)**

,Model,Construct,Strength in the two orders,Pairs,Flip rate,Share of all flips
0,Claude Sonnet 5.5,cogency,Strong in both,712,"0.0% [0.0, 0.0]",0.0%
1,Claude Sonnet 5.5,cogency,Strong in one,192,"2.1% [0.5, 4.2]",2.7%
2,Claude Sonnet 5.5,cogency,Weak in both,821,"17.5% [14.9, 20.3]",97.3%
3,Claude Sonnet 5.5,effectiveness,Strong in both,584,"0.0% [0.0, 0.0]",0.0%
4,Claude Sonnet 5.5,effectiveness,Strong in one,204,"2.5% [0.5, 5.0]",2.9%
5,Claude Sonnet 5.5,effectiveness,Weak in both,937,"17.6% [15.2, 20.1]",97.1%
6,GPT-6.1 Sol,cogency,Strong in both,815,"0.0% [0.0, 0.0]",0.0%
7,GPT-6.1 Sol,cogency,Strong in one,154,"0.0% [0.0, 0.0]",0.0%
8,GPT-6.1 Sol,cogency,Weak in both,756,"12.2% [9.9, 14.5]",100.0%
9,GPT-6.1 Sol,effectiveness,Strong in both,876,"0.0% [0.0, 0.0]",0.0%


... (2 outputs left out)

**X6_construct_gap. Construct gap G = effectiveness percentile - cogency percentile, per response**

,Method,Responses,Mean |G|,|G| >= 1/3
0,Human annotation (H1),690,0.093,12.3%
1,Claude Sonnet 5.5,690,0.097,8.3%
2,GPT-6.1 Sol,690,0.100,8.0%
3,H2,690,0.005,0.0%
4,H3,348,0.064,6.3%


**X7_flagged_excluded. Agreement with the human excluding pairs with a response flagged as non-argumentative or unreadable**

,Model,Construct,"Same winner, all pairs","Same winner, flagged pairs excluded",Flagged pairs
0,Claude Sonnet 5.5,cogency,85.8%,85.1%,143
1,Claude Sonnet 5.5,effectiveness,80.1%,78.8%,143
2,GPT-6.1 Sol,cogency,81.4%,80.6%,143
3,GPT-6.1 Sol,effectiveness,81.9%,81.3%,143


## Part D. Studies 2 and 3 (115 posts)

The main Study 2 and 3 models use the LLM rankings; supplementary tables fit the same models with the human (H1) rankings.

In [9]:
o = v3.copy(); o["state"] = o.behavior_state_star
d = o[o.post_id.isin(MAIN_OPS)]
save(pd.DataFrame([{"Dataset": "All main (115 posts)", "Posts": d.post_id.nunique(), "Responses": len(d),
                    "Delta (D)": int((d.state == "D").sum()), "OP reply, no delta (R*)": int((d.state == "R").sum()),
                    "No OP reply (N)": int((d.state == "N").sum())}]), "S0_sample", "Analysis sample and behavioral states")

D_all = build_dataset(v3, W, inp)
DATA = {"All main (115 OPs)": D_all[D_all.post_id.isin(MAIN_OPS)]}
rows = []
for mod in ("H", "gpt", "claude"):
    D = DATA["All main (115 OPs)"]
    r = D[[f"{mod}_C", f"{mod}_E"]].dropna().corr().iloc[0, 1]
    rows.append({"Method": METHOD_NAMES[mod] if mod != "H" else "Human (H1)", "Correlation c-e": f2(r),
                 "Variance inflation (1/(1-r^2))": f2(1 / (1 - r ** 2))})
save(pd.DataFrame(rows), "S2_collinearity", "Overlap between the cogency and effectiveness predictors (115 posts)")

**S0_sample. Analysis sample and behavioral states**

,Dataset,Posts,Responses,Delta (D),"OP reply, no delta (R*)",No OP reply (N)
0,All main (115 posts),115,690,97,230,363


**S2_collinearity. Overlap between the cogency and effectiveness predictors (115 posts)**

,Method,Correlation c-e,Variance inflation (1/(1-r^2))
0,Human (H1),0.84,3.47
1,GPT-6.1 Sol,0.84,3.42
2,Claude Sonnet 5.5,0.85,3.63


In [10]:
OUTCOMES = {"D": "Delta (D)", "Rstar_vs_N": "OP reply, no delta (R* vs. N)", "D_vs_N": "Delta vs. no reply (D vs. N)", "U": "Score percentile (U)"}

def effect_cells(b, lo, hi, outcome):
    if outcome == "U":
        return f"{50*b:+.1f} [{50*lo:+.1f}, {50*hi:+.1f}]"
    if abs(b) > 15:
        return "separation"
    orf = lambda v: ">999" if np.exp(v / 2) > 999 else ("<0.001" if np.exp(v / 2) < 0.001 else f"{np.exp(v/2):.2f}")
    return f"{orf(b)} [{orf(lo)}, {orf(hi)}]"

def outcome_table(D, c, e, param="separate", weights=False, seed=0):
    res = fit_outcomes(D, c, e, param, weights)
    keys_ = ["c", "e"] if param == "separate" else ["level", "diff"]
    bo = boot_outcomes(D, c, e, B, seed, param, weights,
                       extra=(lambda r: {"angle": angle(r["D"][0][["c", "e"]].values, r["U"][0][["c", "e"]].values)})
                       if param == "separate" else None)
    rows, ps = [], []
    for o_ in OUTCOMES:
        b, V = res[o_]
        stat, p = wald(b.values, V.values, [0, 1])
        row = {"Outcome": OUTCOMES[o_]}
        for k in keys_:
            lo, hi = np.nanpercentile(bo[f"{o_}:{k}"], [2.5, 97.5])
            row[{"c": "Cogency", "e": "Effectiveness", "level": "Level (c+e)/2", "diff": "Difference (e-c)/2"}[k]] = effect_cells(b[k], lo, hi, o_)
        row["Joint test p"] = p; rows.append(row)
    t = pd.DataFrame(rows)
    main = t.Outcome.isin([OUTCOMES["D"], OUTCOMES["Rstar_vs_N"], OUTCOMES["U"]])
    t["Holm p (D, R*, U)"] = ""
    t.loc[main, "Holm p (D, R*, U)"] = [pval(x) for x in holm(t.loc[main, "Joint test p"].values)]
    t["Joint test p"] = t["Joint test p"].map(pval)
    return t, res, bo

D = DATA["All main (115 OPs)"]
S2 = {}; tabs = []
for mod in ("gpt", "claude"):
    t, res, bo = outcome_table(D, f"{mod}_C", f"{mod}_E")
    S2[mod] = (res, bo, t)
    t = t.copy(); t.insert(0, "Model", METHOD_NAMES[mod]); tabs.append(t)
save(pd.concat(tabs, ignore_index=True), "S2_outcomes", "Behavioral outcomes on LLM cogency and effectiveness (25th to 75th within-post percentile), 115 posts",
     "Odds ratios for D, R* vs. N and D vs. N; percentile points for U. 95% post-bootstrap CIs. Covariates: log words, log hours since OP and its square, log response order.")

tabs = []; S2L = {}
for mod in ("gpt", "claude"):
    t, _, _ = outcome_table(D, f"{mod}_C", f"{mod}_E", param="level")
    S2L[mod] = t
    t = t.copy(); t.insert(0, "Model", METHOD_NAMES[mod]); tabs.append(t)
save(pd.concat(tabs, ignore_index=True), "S2_level_difference", "Behavioral outcomes on the shared level and the difference of the two constructs, 115 posts")

tabs = []
for mod in ("gpt", "claude"):
    res = fit_outcomes(D, f"{mod}_C", f"{mod}_E", weights=True)
    bo = boot_outcomes(D, f"{mod}_C", f"{mod}_E", B, 1, weights=True)
    b = res["U"][0]; row = {"Model": METHOD_NAMES[mod]}
    for k, nm_ in (("c", "Cogency"), ("e", "Effectiveness")):
        lo, hi = np.nanpercentile(bo[f"U:{k}"], [2.5, 97.5]); row[nm_] = effect_cells(b[k], lo, hi, "U")
    tabs.append(row)
save(pd.DataFrame(tabs), "S2_score_weighted", "Score model with inverse-probability weights for the outcome-stratified sampling, 115 posts")

tabs = []
for param in ("separate", "level"):
    t, _, _ = outcome_table(D, "H_C", "H_E", param=param)
    t.insert(0, "Parameterization", "cogency and effectiveness" if param == "separate" else "level and difference"); tabs.append(t)
S2H = pd.concat(tabs, ignore_index=True)
save(S2H, "S2_human_supplementary", "Supplementary: the Study 2 models fitted with the human (H1) rankings, 115 posts")

**S2_outcomes. Behavioral outcomes on LLM cogency and effectiveness (25th to 75th within-post percentile), 115 posts**  
_Odds ratios for D, R* vs. N and D vs. N; percentile points for U. 95% post-bootstrap CIs. Covariates: log words, log hours since OP and its square, log response order._

,Model,Outcome,Cogency,Effectiveness,Joint test p,"Holm p (D, R*, U)"
0,GPT-6.1 Sol,Delta (D),"1.01 [0.45, 2.50]","3.41 [1.39, 8.46]",<.001,<.001
1,GPT-6.1 Sol,"OP reply, no delta (R* vs. N)","0.77 [0.42, 1.42]","1.05 [0.54, 2.05]",0.508,0.508
2,GPT-6.1 Sol,Delta vs. no reply (D vs. N),"1.09 [0.39, 3.32]","3.25 [1.14, 12.74]",<.001,
3,GPT-6.1 Sol,Score percentile (U),"+0.4 [-6.0, +6.4]","+7.4 [+2.2, +13.9]",0.003,0.006
4,Claude Sonnet 5.5,Delta (D),"2.24 [1.03, 5.79]","1.34 [0.59, 2.95]",0.001,0.003
5,Claude Sonnet 5.5,"OP reply, no delta (R* vs. N)","0.92 [0.46, 1.91]","0.94 [0.49, 1.82]",0.817,0.817
6,Claude Sonnet 5.5,Delta vs. no reply (D vs. N),"2.19 [0.90, 7.30]","1.41 [0.44, 4.11]",0.007,
7,Claude Sonnet 5.5,Score percentile (U),"+8.2 [+1.7, +15.1]","+1.3 [-4.3, +6.8]",0.006,0.011


**S2_level_difference. Behavioral outcomes on the shared level and the difference of the two constructs, 115 posts**

,Model,Outcome,Level (c+e)/2,Difference (e-c)/2,Joint test p,"Holm p (D, R*, U)"
0,GPT-6.1 Sol,Delta (D),"3.43 [2.14, 6.38]","3.38 [0.59, 15.99]",<.001,<.001
1,GPT-6.1 Sol,"OP reply, no delta (R* vs. N)","0.80 [0.53, 1.22]","1.36 [0.41, 4.51]",0.508,0.508
2,GPT-6.1 Sol,Delta vs. no reply (D vs. N),"3.55 [1.99, 9.09]","2.97 [0.35, 28.37]",<.001,
3,GPT-6.1 Sol,Score percentile (U),"+7.8 [+3.2, +12.3]","+7.0 [-3.2, +19.3]",0.003,0.006
4,Claude Sonnet 5.5,Delta (D),"3.00 [1.61, 6.23]","0.60 [0.11, 2.37]",0.001,0.003
5,Claude Sonnet 5.5,"OP reply, no delta (R* vs. N)","0.86 [0.55, 1.32]","1.02 [0.28, 3.69]",0.817,0.817
6,Claude Sonnet 5.5,Delta vs. no reply (D vs. N),"3.08 [1.59, 7.05]","0.64 [0.07, 4.18]",0.007,
7,Claude Sonnet 5.5,Score percentile (U),"+9.5 [+3.9, +15.0]","-6.8 [-18.7, +3.9]",0.006,0.011


**S2_score_weighted. Score model with inverse-probability weights for the outcome-stratified sampling, 115 posts**

,Model,Cogency,Effectiveness
0,GPT-6.1 Sol,"-3.4 [-10.7, +4.4]","+7.3 [+0.5, +14.3]"
1,Claude Sonnet 5.5,"+3.7 [-3.9, +12.0]","+1.6 [-6.0, +9.4]"


**S2_human_supplementary. Supplementary: the Study 2 models fitted with the human (H1) rankings, 115 posts**

,Parameterization,Outcome,Cogency,Effectiveness,Joint test p,"Holm p (D, R*, U)",Level (c+e)/2,Difference (e-c)/2
0,cogency and effectiveness,Delta (D),"3.34 [1.58, 8.12]","0.91 [0.40, 1.90]",<.001,<.001,NaN,NaN
1,cogency and effectiveness,"OP reply, no delta (R* vs. N)","1.08 [0.50, 2.19]","0.89 [0.44, 1.86]",0.931,0.931,NaN,NaN
2,cogency and effectiveness,Delta vs. no reply (D vs. N),"6.13 [2.15, 22.05]","0.68 [0.24, 1.77]",<.001,,NaN,NaN
3,cogency and effectiveness,Score percentile (U),"+10.8 [+4.0, +17.1]","-2.2 [-7.5, +3.1]",0.004,0.008,NaN,NaN
4,level and difference,Delta (D),NaN,NaN,<.001,<.001,"3.03 [1.70, 5.86]","0.27 [0.06, 1.00]"
5,level and difference,"OP reply, no delta (R* vs. N)",NaN,NaN,0.931,0.931,"0.97 [0.63, 1.46]","0.82 [0.21, 3.65]"
6,level and difference,Delta vs. no reply (D vs. N),NaN,NaN,<.001,,"4.14 [2.20, 10.62]","0.11 [0.01, 0.69]"
7,level and difference,Score percentile (U),NaN,NaN,0.004,0.008,"+8.6 [+3.4, +13.6]","-13.0 [-23.7, -2.0]"


In [11]:
def incremental_rows(D, c, e, label):
    hd = heldout_discrimination(D, c, e)
    piv = {k: hd.pivot(index="post_id", columns="model", values=k) for k in ("auc_D", "rho_U")}
    rows = []
    for k, nm_ in (("auc_D", "Delta: within-post AUC"), ("rho_U", "Score: within-post rank r")):
        P_ = piv[k]; row = {**label, "Metric": nm_}
        for mdl in ("covariates only", "cogency + covariates", "effectiveness + covariates", "both + covariates"):
            row[mdl.replace(" + covariates", "")] = f2(P_[mdl].mean())
        for a_, b_, nm2 in (("both + covariates", "cogency + covariates", "Gain from adding effectiveness"),
                            ("both + covariates", "effectiveness + covariates", "Gain from adding cogency")):
            d_ = (P_[a_] - P_[b_]).dropna().values
            rng = np.random.default_rng(0)
            bs = [rng.choice(d_, len(d_)).mean() for _ in range(B)]
            row[nm2] = f"{d_.mean():+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]"
        rows.append(row)
    return rows

rows = []
for mod in ("gpt", "claude", "H"):
    rows += incremental_rows(D, f"{mod}_C", f"{mod}_E", {"Measurement": METHOD_NAMES[mod] if mod != "H" else "Human (H1)"})
S3I = pd.DataFrame(rows)
save(S3I, "S3_incremental", "Held-out-post discrimination of nested models, 115 posts",
     "Means over held-out posts; gains with 95% bootstrap CIs. Delta AUC uses only posts with a delta. Human row is supplementary.")

rows = []
for mod in ("gpt", "claude"):
    res, bo, _ = S2[mod]
    bD, bU = res["D"][0], res["U"][0]
    a = angle(bD[["c", "e"]].values, bU[["c", "e"]].values)
    lo, hi = np.nanpercentile(bo["angle"], [2.5, 97.5])
    rows.append({"Model": METHOD_NAMES[mod], "Delta model (beta_C, beta_E)": f"({bD['c']:.2f}, {bD['e']:.2f})",
                 "Score model (beta_C, beta_E)": f"({bU['c']:.3f}, {bU['e']:.3f})",
                 "Angle, degrees [95% CI]": f"{a:+.1f} [{lo:+.1f}, {hi:+.1f}]", "Different weighting?": "yes" if lo > 0 or hi < 0 else "no"})
save(pd.DataFrame(rows), "S3_differential", "Differential prediction: do delta and score weight the constructs in the same proportion?")

rows = []
for name, Jr in (("H1, 115 posts", J), ("H3, 58 posts", JREF["H3, 58 posts"]), ("H2, 115 posts", JREF["H2, 115 posts"])):
    P_ = discordance(Jr)
    ops = H3_OPS if name.startswith("H3") else MAIN_OPS
    s = P_[(P_.method == "H") & P_.post_id.isin(ops)]
    rows.append({"Method": f"Human: {name}", "Pairs": len(s), "Different winners for C and E": f"{100*s.Z.mean():.1f}%"})
P_ = discordance(J)
for mod in ("gpt", "claude"):
    s = P_[(P_.method == mod) & P_.post_id.isin(MAIN_OPS)]
    rows.append({"Method": f"{METHOD_NAMES[mod]}, 115 posts", "Pairs": len(s),
                 "Different winners for C and E": f"{100*s.Z.mean():.1f}% (order-split pairs excluded: {100*s.split.mean():.1f}%)"})
save(pd.DataFrame(rows), "S3_discordance", "Pairs on which cogency and effectiveness pick different winners")

**S3_incremental. Held-out-post discrimination of nested models, 115 posts**  
_Means over held-out posts; gains with 95% bootstrap CIs. Delta AUC uses only posts with a delta. Human row is supplementary._

,Measurement,Metric,covariates only,cogency,effectiveness,both,Gain from adding effectiveness,Gain from adding cogency
0,GPT-6.1 Sol,Delta: within-post AUC,0.77,0.76,0.78,0.77,"+0.008 [-0.025, +0.041]","-0.012 [-0.025, +0.000]"
1,GPT-6.1 Sol,Score: within-post rank r,0.55,0.54,0.54,0.54,"-0.001 [-0.021, +0.018]","-0.007 [-0.017, +0.002]"
2,Claude Sonnet 5.5,Delta: within-post AUC,0.77,0.75,0.75,0.73,"-0.019 [-0.035, -0.002]","-0.023 [-0.049, +0.004]"
3,Claude Sonnet 5.5,Score: within-post rank r,0.55,0.55,0.55,0.55,"-0.004 [-0.010, +0.001]","-0.001 [-0.027, +0.023]"
4,Human (H1),Delta: within-post AUC,0.77,0.76,0.76,0.76,"+0.002 [-0.006, +0.012]","+0.002 [-0.029, +0.033]"
5,Human (H1),Score: within-post rank r,0.55,0.56,0.54,0.56,"-0.001 [-0.010, +0.009]","+0.017 [-0.006, +0.040]"


**S3_differential. Differential prediction: do delta and score weight the constructs in the same proportion?**

,Model,"Delta model (beta_C, beta_E)","Score model (beta_C, beta_E)","Angle, degrees [95% CI]",Different weighting?
0,GPT-6.1 Sol,"(0.01, 2.45)","(0.008, 0.149)","+2.8 [-69.5, +67.4]",no
1,Claude Sonnet 5.5,"(1.61, 0.59)","(0.163, 0.026)","+11.0 [-61.3, +76.1]",no


**S3_discordance. Pairs on which cogency and effectiveness pick different winners**

,Method,Pairs,Different winners for C and E
0,"Human: H1, 115 posts",1725,12.2%
1,"Human: H3, 58 posts",870,8.3%
2,"Human: H2, 115 posts",1725,0.5%
3,"GPT-6.1 Sol, 115 posts",1725,9.7% (order-split pairs excluded: 10.1%)
4,"Claude Sonnet 5.5, 115 posts",1725,7.1% (order-split pairs excluded: 16.0%)


## Part D2. Study 3 follow-up: overlap with length, timing and position

How strongly each quality ranking moves with length, timing and position within a post, and how well quality alone predicts the delta on held-out posts. Free to run; no API calls.

In [12]:
# ===== Study 3 follow-up: how much do the quality rankings overlap with length, timing and position? =====
# Free to run: uses only data already loaded above. No API calls.
# Run after Part D (needs D, clogit, covariates, METHOD_NAMES, save, B).

D = DATA["All main (115 OPs)"]
D = D.merge(WH[["post_id", "response_id", "H2_C", "H2_E", "H3_C", "H3_E"]], on=["post_id", "response_id"], how="left")
# (prefix, label, posts). The three-annotator overlap has 58 posts; H1 is repeated on those posts for comparison.
MEAS = [("H", "Human (H1), 115 posts", MAIN_OPS), ("H2", "Human (H2), 115 posts", MAIN_OPS),
        ("H3", "Human (H3), 58 posts", H3_OPS), ("H", "Human (H1), same 58 posts", H3_OPS),
        ("claude", "Claude Sonnet 5.5, 115 posts", MAIN_OPS), ("gpt", "GPT-6.1 Sol, 115 posts", MAIN_OPS)]
FEATS = [("n_words", "Length (words)"), ("hours_since_op", "Hours after the post"), ("response_order", "Reply position")]

def boot_mean(v, seed=0):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    rng = np.random.default_rng(seed)
    bs = [rng.choice(v, len(v)).mean() for _ in range(B)]
    return f"{v.mean():+.2f} [{np.percentile(bs, 2.5):+.2f}, {np.percentile(bs, 97.5):+.2f}]"

# (a) Within each post, how strongly does each ranking move with length, timing and position?
rows = []
for mod, nm, ops in MEAS:
    Dm = D[D.post_id.isin(ops)]
    for con, cnm in (("C", "cogency"), ("E", "effectiveness")):
        col = f"{mod}_{con}"; row = {"Ranking": f"{nm}, {cnm}"}
        for f, fnm in FEATS:
            r = [g[col].rank().corr(g[f].rank()) for _, g in Dm.groupby("post_id") if g[f].nunique() > 1]
            row[fnm] = boot_mean(r)
        rows.append(row)
save(pd.DataFrame(rows), "S3_overlap_controls",
     "Within-post rank correlation between each quality ranking and length, timing and position",
     "Mean over posts of the Spearman correlation within each post; 95% bootstrap CI over posts. Positive = longer / later / further down.")

# (b) Held-out delta AUC: quality alone vs. length, timing and position alone vs. both together.
def heldout_sets(D, c, e, folds=5, seed=0):
    rng = np.random.default_rng(seed)                      # same folds as heldout_discrimination
    ops = np.array(sorted(D.post_id.unique())); rng.shuffle(ops)
    fold_of = {p: i % folds for i, p in enumerate(ops)}
    keep = covariates(D).std() > 1e-9
    def X_of(d, name):
        cov = covariates(d).loc[:, keep]
        q = pd.DataFrame({"c": d[c].values, "e": d[e].values}, index=d.index)
        return {"length only": cov[["log_words"]], "length, timing, position": cov,
                "quality only": q, "quality + length, timing, position": pd.concat([q, cov], axis=1)}[name]
    names = ["length only", "length, timing, position", "quality only", "quality + length, timing, position"]
    out = []
    for k in range(folds):
        tr = D[D.post_id.map(fold_of) != k]; te = D[D.post_id.map(fold_of) == k]
        for name in names:
            b, _ = clogit(tr.y_D.values, X_of(tr, name).values, tr.post_id.values)
            te = te.assign(pD=X_of(te, name).values @ b)
            for pid, g in te.groupby("post_id"):
                pos, neg = g[g.y_D == 1].pD.values, g[g.y_D == 0].pD.values
                if len(pos) and len(neg):
                    out.append({"post_id": pid, "model": name,
                                "auc": np.mean([(p > n) + 0.5 * (p == n) for p in pos for n in neg])})
    return pd.DataFrame(out).pivot(index="post_id", columns="model", values="auc")[names]

rows = []
for mod, nm, ops in MEAS:
    P_ = heldout_sets(D[D.post_id.isin(ops)], f"{mod}_C", f"{mod}_E")
    row = {"Measurement": nm}
    for m in P_.columns:
        row[m] = f"{P_[m].mean():.2f}"
    row["Quality only minus length, timing, position"] = boot_mean(P_["quality only"] - P_["length, timing, position"])
    row["Quality only minus chance (.50)"] = boot_mean(P_["quality only"] - 0.5)
    rows.append(row)
save(pd.DataFrame(rows), "S3_quality_alone",
     "Held-out within-post delta AUC: quality alone, length/timing/position alone, and both",
     "Five-fold cross-validation by post (same fold rule as S3_incremental; the 58-post rows are folded among those posts). Quality = both construct rankings of that measurement. Differences with 95% bootstrap CIs over posts.")

**S3_overlap_controls. Within-post rank correlation between each quality ranking and length, timing and position**  
_Mean over posts of the Spearman correlation within each post; 95% bootstrap CI over posts. Positive = longer / later / further down._

,Ranking,Length (words),Hours after the post,Reply position
0,"Human (H1), 115 posts, cogency","+0.67 [+0.62, +0.72]","-0.04 [-0.10, +0.04]","-0.04 [-0.10, +0.04]"
1,"Human (H1), 115 posts, effectiveness","+0.53 [+0.47, +0.59]","-0.04 [-0.11, +0.04]","-0.04 [-0.11, +0.04]"
2,"Human (H2), 115 posts, cogency","+0.72 [+0.67, +0.77]","-0.04 [-0.12, +0.04]","-0.04 [-0.12, +0.04]"
3,"Human (H2), 115 posts, effectiveness","+0.72 [+0.67, +0.77]","-0.03 [-0.11, +0.05]","-0.03 [-0.11, +0.05]"
4,"Human (H3), 58 posts, cogency","+0.67 [+0.60, +0.74]","-0.03 [-0.13, +0.08]","-0.03 [-0.13, +0.08]"
5,"Human (H3), 58 posts, effectiveness","+0.66 [+0.59, +0.73]","-0.07 [-0.17, +0.04]","-0.07 [-0.17, +0.04]"
6,"Human (H1), same 58 posts, cogency","+0.65 [+0.58, +0.73]","-0.06 [-0.15, +0.04]","-0.06 [-0.15, +0.04]"
7,"Human (H1), same 58 posts, effectiveness","+0.51 [+0.43, +0.60]","-0.09 [-0.19, +0.02]","-0.09 [-0.19, +0.02]"
8,"Claude Sonnet 5.5, 115 posts, cogency","+0.67 [+0.62, +0.71]","-0.02 [-0.08, +0.06]","-0.02 [-0.08, +0.06]"
9,"Claude Sonnet 5.5, 115 posts, effectiveness","+0.58 [+0.52, +0.64]","+0.00 [-0.06, +0.08]","+0.00 [-0.06, +0.08]"


**S3_quality_alone. Held-out within-post delta AUC: quality alone, length/timing/position alone, and both**  
_Five-fold cross-validation by post (same fold rule as S3_incremental; the 58-post rows are folded among those posts). Quality = both construct rankings of that measurement. Differences with 95% bootstrap CIs over posts._

,Measurement,length only,"length, timing, position",quality only,"quality + length, timing, position","Quality only minus length, timing, position",Quality only minus chance (.50)
0,"Human (H1), 115 posts",0.73,0.77,0.75,0.76,"-0.02 [-0.07, +0.03]","+0.25 [+0.19, +0.30]"
1,"Human (H2), 115 posts",0.73,0.77,0.74,0.76,"-0.03 [-0.08, +0.02]","+0.24 [+0.18, +0.29]"
2,"Human (H3), 58 posts",0.69,0.72,0.69,0.74,"-0.03 [-0.10, +0.05]","+0.19 [+0.11, +0.26]"
3,"Human (H1), same 58 posts",0.69,0.72,0.71,0.73,"-0.01 [-0.09, +0.08]","+0.21 [+0.14, +0.28]"
4,"Claude Sonnet 5.5, 115 posts",0.73,0.77,0.73,0.73,"-0.04 [-0.10, +0.03]","+0.23 [+0.18, +0.28]"
5,"GPT-6.1 Sol, 115 posts",0.73,0.77,0.75,0.77,"-0.02 [-0.08, +0.05]","+0.25 [+0.20, +0.31]"


## Files

Every table is written to `results/tables/` as `.csv` and `.tex`; `all_tables.tex` collects them.

In [13]:
for f in sorted(T.glob("*.csv")):
    print(f.relative_to(ROOT))

results/tables/A1_annotators.csv
results/tables/A2_human_agreement.csv
results/tables/A3_human_reliability.csv
results/tables/B0_reliability.csv
results/tables/B1_pairwise.csv
results/tables/B2_strength.csv
results/tables/B3_order.csv
results/tables/C1_study1_by_reference.csv
results/tables/C2_pairwise_by_reference.csv
results/tables/S0_sample.csv
results/tables/S1_associations.csv
results/tables/S1_contrasts.csv
results/tables/S1_lambda.csv
results/tables/S1_model_model.csv
results/tables/S1_robustness.csv
results/tables/S2_collinearity.csv
results/tables/S2_human_supplementary.csv
results/tables/S2_level_difference.csv
results/tables/S2_outcomes.csv
results/tables/S2_score_weighted.csv
results/tables/S3_differential.csv
results/tables/S3_discordance.csv
results/tables/S3_incremental.csv
results/tables/S3_overlap_controls.csv
results/tables/S3_quality_alone.csv
results/tables/X1_strength_and_position.csv
results/tables/X2_paired_models.csv
results/tables/X4_flips_by_model_strength.csv